<center><font size=6>Airlines Q&A Bot</font></center>

## Problem Statement

### Business Context

Modern organizations operate in an environment where internal policies have grown increasingly complex, covering areas such as leaves, reimbursements, travel, hybrid work norms, performance management, compliance, and more. These policies are commonly documented in static, lengthy HR handbooks or PDFs, which employees often find difficult to understand, search, or apply in their daily work scenarios.

As a result:

* HR teams are burdened with repetitive queries about standard policies, diverting their focus from strategic initiatives.
* Employees struggle with confusion or non-compliance, as the dense and static format of policy documents obscures the necessary guidance.
* Both sides suffer from reduced productivity, HR spends excessive time addressing routine questions, while employees experience delays in obtaining the information they need to perform efficiently.

Addressing these challenges requires modern HR systems that centralize policy information, simplify access, and deliver clear, actionable insights. By leveraging technology to streamline policy communication and automate routine queries, organizations can enhance clarity, boost compliance, and ultimately improve overall operational efficiency.

### Objective

The goal is to develop a prototype that demonstrates how Natural Language Processing (NLP), powered by Retrieval-Augmented Generation (RAG), can help employees efficiently query company HR policies and receive accurate, context-aware, and easily understandable answers. Specifically, the system aims to:

* Answer employee questions by retrieving relevant content from official HR handbooks and policy documents.
* Handle ambiguous queries and follow-up questions by clarifying intent and distinguishing between similar policy categories (e.g., sick leave versus casual leave).
* Personalize responses based on role, location, or department, acknowledging that policies may differ (e.g., field staff versus HQ).
* Increase trust and compliance by citing sources (document name, section, and clause) for each response.

**Questions to Answer**

1. What are the effects on the benefits I receive if my probation is extended?
2. There has been a demise in my family last night, and I need to attend the last rites. How should I inform the office, and will I be granted leave?
3. What should I do if I notice suspected harassment with my female colleague?

### Data Description

The employee handbook is an internal reference published by Flykite Airlines that outlines a wide range of workplace policies, guidelines, and procedures for staff. The handbook is provided in PDF format and serves as a comprehensive resource for employees across different roles within the airline.

## Installing and Importing Necessary Libraries and Dependencies

In [1]:
# Installation for GPU llama-cpp-python
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.1.85 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 28.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 247.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 240.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 255.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.6 which is incompatible.


In [2]:
# For installing the libraries & doumloading models from HF hub
!pip install huggingface_hub==0.35.0 pandas==2.2.2 tiktoken==0.11.0 pymupdf==1.26.4 langchain==0.3.27 langchain-community==0.3.29 langchain_experimental==0.3.4 chromadb==1.1.0 sentence-transformers==5.1.0 numpy==2.3.3 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 563.4/563.4 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 483.4/483.4 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2

In [1]:
# Libraries for processing dataframes, text
import json, os
import tiktoken
import pandas as pd

# Libraries for loading data, chunking, Embedding and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

# Library for downloading and loading llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

### Question Answering using LLM

#### Downloading and Loading the model

In [2]:
# Loading model
model_name_or_path = 'TheBloke/Mistral-7B-Instruct-v0.2-GGUF'
model_basename = 'mistral-7b-instruct-v0.2.Q6_K.gguf'       # model in gguf format

In [3]:
# Using hf_hub_download to download a model from the Hugging face model hub
model_path = hf_hub_download(
    repo_id = model_name_or_path,         # repo id specifies the model name or path in Hugging face repository
    filename = model_basename             # specifies name of the file to download
)

mistral-7b-instruct-v0.2.Q6_K.gguf:   0%|          | 0.00/5.94G [00:00<?, ?B/s]

In [4]:
# Initialize Llama model
llm = Llama(
    model_path = model_path,
    n_ctx = 4096,                   # context window
    n_batch = 512,                  # Should be between 1 and n_ctx
    n_gpu_layers = 38,              # GPU VRAM pool
    seed = 42
)

AVX = 1 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


#### Function to define model parameters for generating response

In [5]:
# Function to generate response
def response(query, max_tokens=1024, temperature=0, repeat_penalty=1.2, top_p=0.95, top_k=50):
  model_output = llm(
      prompt=query,                           # User query
      max_tokens = max_tokens,                # Number of tokens the response should be generated by the model
      temperature = temperature,              # Controls the randomness of the response to be generated
      top_p = top_p,                          # Controls the diversity of the generated response by establishing cummulative probability cut-off for token selection
      top_k = top_k,                          # Maximum number of most likely next tokens to be considered
      repeat_penalty = repeat_penalty,        # Penalizes repeated tokens to improve text diversity
      stop = ['INST'],
      echo = False
  )

  return model_output['choices'][0]['text']

#### List of Questions

In [6]:
# List of questions
user_queries = [
    'What are the effects on the benefits I receive if my probation is extended?',
    'There has been a demise in my family last night, and I need to attend the last rites. How should I inform the office, and will I be granted leave?',
    'What should I do if I notice suspected harassment with my female colleague?'
]

##### Query 1: What are the effects on the benefits I receive if my probation is extended?

In [7]:
# get the response for the user query
query_1 = user_queries[0]
response_1 = response(query=query_1)
print(response_1)



If your probation is extended, it may have several impacts on the benefits you receive. Here's a brief overview:

1. Social Security Disability Insurance (SSDI) and Supplemental Security Income (SSI): Probation itself does not directly affect SSDI or SSI eligibility as long as you are complying with its terms. However, if your probation is extended due to a violation, it could potentially impact your benefits in several ways:
   - Suspension of Benefits: If you violate the conditions of your probation and face sanctions such as incarceration or community service, Social Security Administration (SSA) may suspend your disability benefits during that time. However, once you are released from confinement or complete your community service, your benefits will be reinstated automatically if you continue to meet eligibility requirements.
   - Reapplication: If your probation is extended due to a violation and results in an incarceration period of 30 days or more, SSA may consider it as a br

Observation

* The user's query is strictly an internal HR policy regarding the impact on the benefits an employee receive if probation is extended. But the response generated is hallucinated between two domains i.e., criminal and corporate due to probation term, but here the user query is related to corporate domain.

##### Query 2: There has been a demise in my family last night, and I need to attend the last rites. How should I inform the office, and will I be granted leave?

In [8]:
# get the response for the user query
query_2 = user_queries[1]
response_2 = response(query=query_2)
print(response_2)

Llama.generate: prefix-match hit




If you are an employee who needs to take time off for attending a funeral or last rites of a close family member, it is essential to follow proper procedures to ensure that your absence from work is authorized and communicated effectively to your employer. Here's what you can do:

1. Notify your manager or supervisor as soon as possible: Inform them about the unfortunate event and request for leave. Provide details such as the date of the funeral, duration of your absence, and any other relevant information that they may need to know. Be respectful and professional in your communication.
2. Follow company policies regarding bereavement leave: Check with your HR department or employee handbook about the company's policy on bereavement leave. Find out if there are specific procedures you need to follow, such as providing documentation or completing a form. Make sure that you comply with these requirements to ensure that your absence is authorized and that you will be paid for the time 

Observation

* The response is completely adhered to the employee query, outlies general best practices that apply to most of the companies. However, it fails to act as a true internal HR bot due to a point i.e., Follow company policies which is not refering to the internal HR policies.

##### Query 3: What should I do if I notice suspected harassment with my female colleague?

In [9]:
# get the response for the user query
query_3 = user_queries[2]
response_3 = response(query=query_3)
print(response_3)

Llama.generate: prefix-match hit




If you suspect that a female colleague is being subjected to unwelcome and inappropriate behavior, it's important to take action. Here are some steps you can take:

1. Listen actively: Allow your colleague to express her concerns without interrupting or judging her. Make sure she feels heard and understood.
2. Offer support: Let her know that you believe her and offer any assistance she may need, such as reporting the incident to HR or management.
3. Encourage her to report it: If your colleague is comfortable doing so, encourage her to formally report the harassment through the appropriate channels in your organization. This could be HR, a supervisor, or another designated person or department responsible for handling complaints.
4. Document any evidence: If you have witnessed any incidents of harassment yourself, document them as thoroughly as possible, including dates, times, locations, and any relevant details. Keep copies of emails, texts, or other communications related to the 

Observation

* The answer fully complies with the employee's inquiry and outlines broad best practices that are applicable to the majority of companies. However, it says, "Follow your organization's policies," which does not refer to the internal HR policies, it is unable to function as a true internal HR bot but functioning more like an external advisor.

#### Evaluating the response generated

In [10]:
# System message for evaluating groundedness
groundedness_rater_system_message = """
You are tasked to rate an AI-generated answer to a question posed by the employee.
You will be presented with a question, the context used to generate the answer, and the generated answer itself.
In the input, the question begins with ###Question, the context begins with ###Context, and the AI-generated answer begins with ###Answer.

Evaluation Criteria:
Your task is to judge the extent to which the metric is followed by the answer.
Rate it 1 - The metric is not followed at all (The answer relies entirely on outside knowledge or hallucinations)
Rate it 2 - The metric is followed only to a limited extent
Rate it 3 - The metric is followed to a good extent
Rate it 4 - The metric is followed mostly
Rate it 5 - The metric is followed completely (Every single fact in the answer is explicitly backed by the context)

Metric:
Groundedness: The answer must be strictly and exclusively derived from the information presented in the ###Context. No assumptions, extrapolations, or outside knowledge should be introduced.

Instructions:
1. First, write down the explicit steps needed to verify the truthfulness of the answer against the provided context.
2. Provide a step-by-step alignment check verifying whether the generated claims match the source context text.
3. Evaluate the extent to which the metric is followed.
4. Conclude by providing your final evaluation exactly matching the structural format requested below.

Please note: You must format the very end of your response using the exact structure below. Do not use structural headers like '###Question' or '###Context' in your final explanation.

[SCORE]: Enter the single integer rating here (1, 2, 3, 4, or 5)
[EXPLANATION]: Provide your brief explanation here.
"""

Observations

* This prompt is designed to evaluate the groundedness of the response generated by AI i.e., how well the response generated is relevant to the context provided. It asks LLM to compare answers with the context and rate it on a scale 1 to 5.
* 1 indicates the answer is not derived from the context, while 5 indicates the answer is completely derived from the context. This is very useful to determine whether the model is hallucinating or it is providing relevant response grounded to the context retrieved from the document.

In [11]:
# System message for evaluating relevance
relevance_rater_system_message = """
You are tasked to rate the AI generated answer to the question posed by the employee.
You will be presented with a question, the context used by the AI system, and the AI-generated answer.
In the input, the question begins with ###Question, the context begins with ###Context, while the AI-generated answer begins with ###Answer.

Evaluation Criteria:
Your task is to judge the extent to which the metric is followed by the answer.
Rate it 1 - The metric is not followed at all (The answer completely misses the intent of the question)
Rate it 2 - The metric is followed only to a limited extent
Rate it 3 - The metric is followed to a good extent
Rate it 4 - The metric is followed mostly
Rate it 5 - The metric is followed completely (The answer directly, fully, and comprehensively resolves the question)

Metric:
Relevance: Measures how well the answer addresses the main aspects of the question. Consider whether all and only important aspects of the question are contained in the answer.

Instructions:
1. First, write down the explicit steps needed to evaluate the answer against the question constraints.
2. Provide a step-by-step explanation of how the answer addresses the core parts of the question.
3. Next, evaluate the overall extent to which the metric is followed.
4. Conclude by providing your final evaluation exactly matching the structural format requested below.

Please note: You must format the very end of your response using the exact structure below. Do not use structural headers like '###Question' or '###Answer' in your final explanation.

[SCORE]: Enter the single integer rating here (1, 2, 3, 4, or 5)
[EXPLANATION]: Provide your brief explanation here.
"""

Observations

* This prompt is designed to evaluate the relevance of the response generated by AI i.e., how well the response generated is relevant to the question asked by the user. It asks LLM to compare answers and the questions and rate it on a scale 1 to 5.
* 1 indicates the answer is not relevant to the question, while 5 indicates the answer is relevant to the user question, also addressing every aspect of the query. This ensure that the output is not only accurate but also addressing the user question completely.

In [12]:
# user template
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [13]:
# Function for evaluating the responses generated by AI
def eval_grounded_relevance(question, context, answer, max_tokens=1024, temperature=0, top_p=0.95, top_k=50, repeat_penalty=1.2):
  """
  A function to evaluate the groundedness and relevance of the response for the user query
  Key inputs required
  user_query: Question posed by the user
  response: Response generated by the llm
  context: Context used by the llm to generate the response
  """

  # Groundedness evaluation
  # If no context or empty context is provided else device groundedness prompt
  if not context or str(context).strip() == "":
      # Output of groundedness, if context is not presented
      final_grounded_response = "Score: 1 - Evaluation bypassed because ###Context is completely empty. No data was provided to ground the answer."
  else:
      # Combine the user and system message to create a groundedness prompt
      groundedness_prompt = f"""[INST]{groundedness_rater_system_message}
                        {'user'}: {user_message_template.format(question=question, context=context, answer=answer)}
                        [/INST]"""

      # Evaluating for groundedness
      grounded_response = llm(
          prompt = groundedness_prompt,           # Prompt
          max_tokens = max_tokens,                # Number of tokens the response should be generated by the model
          temperature = temperature,              # Controls the randomness of the response to be generated
          top_p = top_p,                          # Controls the diversity of the generated response by establishing cummulative probability cut-off for token selection
          top_k = top_k,                          # Maximum number of most likely next tokens to be considered
          repeat_penalty = repeat_penalty,        # Penalizes repeated tokens to improve text diversity
          stop = ['INST'],
          echo = False
          )

      # Output of groundedness, if context is presented
      final_grounded_response = grounded_response['choices'][0]['text']

  # Combine the user and system message to create a relevance prompt (context is not needed for relevance evaluation)
  relevance_prompt = f"""[INST]{relevance_rater_system_message}
                    {'user'}: {user_message_template.format(question=question, context=context, answer=answer)}
                    [/INST]"""

  # Evaluating for relevance
  relevance_response = llm(
      prompt = relevance_prompt,              # Prompt
      max_tokens = max_tokens,                # Number of tokens the response should be generated by the model
      temperature = temperature,              # Controls the randomness of the response to be generated
      top_p = top_p,                          # Controls the diversity of the generated response by establishing cummulative probability cut-off for token selection
      top_k = top_k,                          # Maximum number of most likely next tokens to be considered
      repeat_penalty = repeat_penalty,        # Penalizes repeated tokens to improve text diversity
      stop = ['INST'],
      echo = False
      )

  # Return the evaluation output for groundedness and relevance
  return final_grounded_response, relevance_response['choices'][0]['text']

##### Query 1: What are the effects on the benefits I receive if my probation is extended?

In [14]:
# Evaluating the user query
ground, rel = eval_grounded_relevance(question=query_1, context="", answer=response_1)
print("Groundedness Evaluation")
print(ground, end='\n\n')
print("Relevance Evaluation")
print(rel)

Llama.generate: prefix-match hit


Groundedness Evaluation
Score: 1 - Evaluation bypassed because ###Context is completely empty. No data was provided to ground the answer.

Relevance Evaluation
 [5]: The AI-generated answer follows the metric completely by directly addressing all main aspects of the question and providing comprehensive information on the effects of extending probation on different types of benefits. It covers Social Security Disability Insurance (SSDI), Supplemental Security Income (SSI), Medicaid, Food Assistance Programs (SNAP), Housing Assistance Programs, Temporary Assistance for Needy Families (TANF), Unemployment Insurance, Veterans Benefits, Pension Benefits, Workers' Compensation, and other benefits. The answer explains how probation extensions could potentially impact eligibility or receipt of these benefits due to violations or incarceration, as well as the potential for reinstatement upon release from confinement or continued compliance with program requirements.


Observations

* For evaluating groundedness, the context should be presented, since it is not presented, the score was resulted as 1.
* For relevance, the metric is completely followed resulting the rating as 5 i.e., almost addressing main aspects of the query in the response generated by llm.

##### Query 2: There has been a demise in my family last night, and I need to attend the last rites.How should I inform the office, and will I be granted leave?

In [15]:
# Evaluating the user query
ground, rel = eval_grounded_relevance(question=query_2, context="", answer=response_2)
print("Groundedness Evaluation")
print(ground, end='\n\n')
print("Relevance Evaluation")
print(rel)

Llama.generate: prefix-match hit


Groundedness Evaluation
Score: 1 - Evaluation bypassed because ###Context is completely empty. No data was provided to ground the answer.

Relevance Evaluation
 [5]: The AI-generated answer follows the metric completely as it directly and comprehensively addresses all important aspects of the question regarding informing the office about attending last rites, following company policies, providing proof of the event, making arrangements for workload coverage, staying in touch with the office during leave, returning to work promptly, and offering support to colleagues. The answer provides explicit steps that an employee can take to ensure a smooth process when requesting bereavement leave.


Observations

* The context should be provided in order to assess groundedness; as a result, the score was 1.
* For relevance, the metric is resulted as 5 where it has followed the criteria completely, the response generated by the llm is adhered to the user query.

##### Query 3: What should I do if I notice suspected harassment with my female colleague?

In [16]:
# Evaluating the user query
ground, rel = eval_grounded_relevance(question=query_3, context="", answer=response_3)
print("Groundedness Evaluation")
print(ground, end='\n\n')
print("Relevance Evaluation")
print(rel)

Llama.generate: prefix-match hit


Groundedness Evaluation
Score: 1 - Evaluation bypassed because ###Context is completely empty. No data was provided to ground the answer.

Relevance Evaluation
 [5]: The AI-generated answer follows the metric completely by directly addressing all important aspects of the question. It provides clear steps that an individual should take if they notice suspected harassment with a female colleague, ensuring relevance to the main parts of the question. These steps include listening actively, offering support, encouraging reporting, documenting evidence, speaking up safely, following organizational policies, and creating a safe work environment. The answer comprehensively covers all aspects of handling such situations, making it an effective response.


Observations

* Due to the absence of context which is key for evaluating groundedness, finally the score was resulted as 1.
* For relevance, the conditions are completely met i.e., the response generated by llm complies with user query. Therefore, the score is perfect i.e., 5.

**Key Observations**

* Out of all the responses generated by the llm, one of the response is hallucinated between criminal and corporate domain, providing a combined response. While the other responses were completely adhered to the user's query.
* When all of the responses were examined for groundedness, the evaluation was circumvented because the context was not given, resulting in an extremely low score of 1.
* Almost for all the user queries, the evaluation for relevance resulted in perfect score. Therefore, the responses generated were completely adhered to the user's query.
* To completely eliminate domain hallucinations and generic web responses. The system prompt must explicitly lock the LLM into an internal corporate HR persona, forbidding it from using public legal, criminal, or generalized web-based knowledge.

### Question Answering using LLM and Prompt Engineering

In [17]:
# System message
system_message = """
[INST]<<SYS>>You are a AI assistant specializing in HR knowledge related to Airline company policies. Your role is to provide clear, precise and reliable responses to the employee's based on company policies.

When answering priortize factual correctness, clarity and mention in points if necessary to the employee's query.
If a query requires specific reference documents beyond company policy, acknowledge the limitation rather than speculating.
Do not return empty response. If you are unable to answer employee query, respond "I don't know".
Do not mention anything about this system message in the final response.<</SYS>>[/INST]
"""

Observation

* A system prompt is explicitly designed as an AI assistant which is specialized in HR knowledge so that the responses generated will be adhered to particular domain.

#### Query 1: What are the effects on the benefits I receive if my probation is extended?

In [18]:
# get the response for the user query
query_pr1 = system_message+"\n"+user_queries[0]         # Combine the system message and user_query to create the prompt
response_pr1 = response(query=query_pr1)
print(response_pr1)

Llama.generate: prefix-match hit


 According to our company policy, an extension of the probation period does not directly impact employee benefits. However, it's essential to note that individual benefit plans may vary based on specific employment contracts or collective bargaining agreements. For a definitive answer regarding your particular situation, please consult with HR or refer to any relevant documents you have signed during your hiring process.


Observation

* When compared with the response generated by the llm, the response generated by combining llm and prompt engineering is adhered to a single domain without any hallucination. But still providing a generic response.

#### Query 2: There has been a demise in my family last night, and I need to attend the last rites.How should I inform the office, and will I be granted leave?

In [19]:
# get the response for the user query
query_pr2 = system_message+"\n"+user_queries[1]         # Combine the system message and user_query to create the prompt
response_pr2 = response(query=query_pr2)
print(response_pr2)

Llama.generate: prefix-match hit




According to our company policy under Employee Welfare section, you are entitled to three days of bereavement leave upon the death of an immediate family member. You must provide a valid death certificate or obituary notice as proof. To notify your team about your absence, kindly inform your manager and HR department at the earliest convenience via email with subject line "Request for Bereavement Leave." Your colleagues can cover your responsibilities during this period.


Observation

* When compared to the response produced by the llm for the same employee inquiry, the answer is accurate but still offers a broad response without citing internal corporate rules.

#### Query 3: What should I do if I notice suspected harassment with my female colleague?

In [20]:
# get the response for the user query
query_pr3 = system_message+"\n"+user_queries[2]         # Combine the system message and user_query to create the prompt
response_pr3 = response(query=query_pr3)
print(response_pr3)

Llama.generate: prefix-match hit


 Our company policy strictly prohibits any form of discrimination or harassment. You are encouraged to report such incidents immediately to your supervisor, HR representative, or through our anonymous reporting hotline. The company will conduct a thorough investigation and take appropriate action based on the findings. If you require further information, please refer to our Employee Handbook under "Discrimination and Harassment" section.


Observation

* The response is in line with the employee's inquiry, although it seems generic and is not referring any specific company policy.

#### Evaluating the response generated

##### Query 1: What are the effects on the benefits I receive if my probation is extended?

In [21]:
# Evaluating the user query
ground, rel = eval_grounded_relevance(question=query_1, context="", answer=response_pr1)
print("Groundedness Evaluation")
print(ground, end='\n\n')
print("Relevance Evaluation")
print(rel)

Llama.generate: prefix-match hit


Groundedness Evaluation
Score: 1 - Evaluation bypassed because ###Context is completely empty. No data was provided to ground the answer.

Relevance Evaluation
 To evaluate the relevance of the AI-generated answer to the user's question, we will follow these steps:
1. Identify the main aspects of the question. In this case, the question asks about the effects on benefits when probation is extended.
2. Determine how well each part of the answer addresses the identified aspects of the question.
3. Evaluate the overall relevance based on whether all important parts are covered and no irrelevant information is included.

The AI-generated answer states that an extension of the probation period does not directly impact employee benefits but encourages consulting HR or relevant documents for a definitive answer regarding individual situations. This response addresses both aspects of the question: the potential effects on benefits (no direct impact) and the importance of considering specific e

Observations

* For evaluating groundedness, the context should be presented, since it is not presented, the score was resulted as 1.
* For relevance, the metric is mostly followed resulting the rating as 4, since there is no definite response and redirecting the employee to consult HR or check for relevant documents.

##### Query 2: There has been a demise in my family last night, and I need to attend the last rites.How should I inform the office, and will I be granted leave?

In [22]:
# Evaluating the user query
ground, rel = eval_grounded_relevance(question=query_2, context="", answer=response_pr2)
print("Groundedness Evaluation")
print(ground, end='\n\n')
print("Relevance Evaluation")
print(rel)

Llama.generate: prefix-match hit


Groundedness Evaluation
Score: 1 - Evaluation bypassed because ###Context is completely empty. No data was provided to ground the answer.

Relevance Evaluation
 3: The metric is followed to a good extent

Explanation:
The AI-generated answer directly addresses the main aspects of the question by providing information on how to inform the office about the need for leave due to a family demise and confirming that such leave is granted under company policy. However, it does not explicitly state whether the employee will be guaranteed the leave or not, only implying it based on the mention of bereavement leave entitlement in the company policy.

The answer also provides clear instructions for how to communicate this absence to both the manager and HR department using email with a specific subject line. This helps ensure that proper notification is made within the organization while the employee attends to their family obligations. Overall, the AI-generated response effectively addresses th

Observations

* The context should be provided in order to assess groundedness; as a result, the score was 1.
* For relevance, the metric is followed to a good extent i.e., some part of the response generated is complied with employee query, while it could be more precise by addressing for how long employee can stay away from work or providing contact information for the HR department if it's not already known to the employee. Therefore, the score is 3.

##### Query 3: What should I do if I notice suspected harassment with my female colleague?

In [23]:
# Evaluating the user query
ground, rel = eval_grounded_relevance(question=query_3, context="", answer=response_pr3)
print("Groundedness Evaluation")
print(ground, end='\n\n')
print("Relevance Evaluation")
print(rel)

Llama.generate: prefix-match hit


Groundedness Evaluation
Score: 1 - Evaluation bypassed because ###Context is completely empty. No data was provided to ground the answer.

Relevance Evaluation
 To evaluate the answer against the question constraints, I would follow these steps:
1. Identify if the answer directly addresses the aspects of the question related to suspected harassment towards a female colleague.
2. Check if the answer provides any actionable steps for reporting such incidents within the company.
3. Evaluate if the answer mentions the importance of immediate reporting and investigation by the company.
4. Consider whether the answer refers to any resources or further information available for employees regarding discrimination and harassment policies.

The AI-generated answer directly addresses the aspects of the question related to suspected harassment towards a female colleague, as it states that "You are encouraged to report such incidents immediately." It also provides actionable steps by suggesting rep

Observations

* Due to the absence of context which is key for evaluating groundedness, finally the score was resulted as 1.
* For relevance, the conditions were met completely i.e., the response generated is adhered with employee query. Therefore, the score resulted as 5.

**Key observations**

* The llm combined with prompt engineering performed better when compared with the responses generated by the llm, since there are no hallucinated answers and the bot is completely acting as a corporate HR bot.
* Because the context was not provided, the evaluation was circumvented when all of the responses were checked for groundedness, yielding an incredibly low score of 1.
* Nearly all user inquiries are evaluated for relevancy, but not all the responses received a perfect score because the answers cite higher authorities, such as HR or supervisors, not providing contact details of HR and pointing to refer the employee handbook rather than acting like an HR bot.
* Therefore, for the llm to function as an HR bot, the context must be introduced in order to fully resolve the problem of groundedness and relevance i.e., citation of higher authority, providing HR contact details and not pointing to refer a particular document.

### Data Preparation for RAG

#### Loading the Data

In [24]:
# Mount drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [25]:
# Get the PDF path
path = '/content/drive/MyDrive/AI & ML Course/12. Capstone Project/Dataset - Flykite Airlines_ HRP.pdf'

# Load the PDF data
pdf_data = PyMuPDFLoader(path)
air_hr = pdf_data.load()

#### Data Overview

##### Checking first 5 pages

In [26]:
# Looping through docs to check first 5 pages
for i in range(5):
  print(f'Page Number: {i+1}', end='\n')
  print(air_hr[i].page_content, end='\n\n')

Page Number: 1
Flykite Airlines: Human Resources Policy 
Handbook 
Introduction 
Flykite Airlines is dedicated to cultivating an organizational culture that synergizes 
operational excellence with a supportive, equitable, and legally compliant workplace 
environment across all departments and employee levels. This document establishes 
an exhaustive framework comprising all human resource policies currently in effect. All 
provisions are subject to amendment, interpretation, or rescindment at the sole 
discretion of the Human Resources and Legal departments. In the event of ambiguities 
or conflicting interpretations, the official determinations by these departments shall 
prevail and govern subsequent actions. 
1. Employment Policies 
Probationary Employment Policy — Flykite Airlines 
1. Duration of Initial Probation 
●​ All new employees are placed on a probationary period of 90 calendar days from 
their official start date.​
 
●​ For technical, safety-critical, or senior management 

Observation

* By examining the first 5 pages of the document, it can be infered that it has been loaded perfectly stating the HR policies of Flykite Airlines company.

##### Checking total pages

In [27]:
# Number of pages
len(air_hr)

14

Observation

* The HR policy handbook contains almost 14 pages.

#### Data Chunking

##### Character Text Splitter

In [28]:
# Chunking the document
char_text_splitter = CharacterTextSplitter(
    separator = '',                            # Document seperator
    chunk_size = 1000,                         # Maximum chunk size
    chunk_overlap = 20,                        # Number of tokens that overlap between the adjacent chunks
    strip_whitespace = False                   # Remove extra whitespaces
)

# Applying the text_splitter method on the pdf file
char_doc_chunks = pdf_data.load_and_split(char_text_splitter)

In [29]:
# Number of chunks the document is divided
len(char_doc_chunks)

26

Observation

* The handbook has been divided into 26 chunks.

In [30]:
# Checking a few document chunks from character chunking
print('Chunk 0:\n', char_doc_chunks[0].page_content, end='\n\n')
print('Chunk 1:\n', char_doc_chunks[1].page_content, end='\n\n')
print('Chunk 13:\n', char_doc_chunks[13].page_content, end='\n\n')
print('Chunk 14:\n', char_doc_chunks[14].page_content, end='\n\n')
print('Chunk 24:\n', char_doc_chunks[24].page_content, end='\n\n')
print('Chunk 25:\n', char_doc_chunks[25].page_content, end='\n\n')

Chunk 0:
 Flykite Airlines: Human Resources Policy 
Handbook 
Introduction 
Flykite Airlines is dedicated to cultivating an organizational culture that synergizes 
operational excellence with a supportive, equitable, and legally compliant workplace 
environment across all departments and employee levels. This document establishes 
an exhaustive framework comprising all human resource policies currently in effect. All 
provisions are subject to amendment, interpretation, or rescindment at the sole 
discretion of the Human Resources and Legal departments. In the event of ambiguities 
or conflicting interpretations, the official determinations by these departments shall 
prevail and govern subsequent actions. 
1. Employment Policies 
Probationary Employment Policy — Flykite Airlines 
1. Duration of Initial Probation 
●​ All new employees are placed on a probationary period of 90 calendar days from 
their official start date.​
 
●​ For technical, safety-critical, or senior management roles

Observation

* The character text splitter is very simple and easier method but very rigid by not considering the structure of the sentence i.e., it is abruptly ending the word without completion, say for chunk_0 it was ended with the word 'probatio' (which doesn't have any meaning), while for the chunk_1 it started with the sentence 'ment roles, probation' though there is a overlap the sharing of context is missing.

##### Recursive Character Chunking based on Token Counts

In [31]:
# Chunking the document
rec_char_text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name = 'cl100k_base',
    chunk_size = 100,                         # Maximum chunk size
    chunk_overlap = 20                        # Number of tokens that overlap between the adjacent chunks
)

# Applying the text_splitter method on the pdf file
rec_char_doc_chunks = pdf_data.load_and_split(rec_char_text_splitter)

In [32]:
# Number of chunks the document is divided
len(rec_char_doc_chunks)

57

Observation

* The handbook has been divided into 57 chunks.

In [33]:
# Checking a few document chunks from recursive character chunking
print('Chunk 0:\n', rec_char_doc_chunks[0].page_content, end='\n\n')
print('Chunk 1:\n', rec_char_doc_chunks[1].page_content, end='\n\n')
print('Chunk 13:\n', rec_char_doc_chunks[13].page_content, end='\n\n')
print('Chunk 14:\n', rec_char_doc_chunks[14].page_content, end='\n\n')
print('Chunk 24:\n', rec_char_doc_chunks[24].page_content, end='\n\n')
print('Chunk 25:\n', rec_char_doc_chunks[25].page_content, end='\n\n')

Chunk 0:
 Flykite Airlines: Human Resources Policy 
Handbook 
Introduction 
Flykite Airlines is dedicated to cultivating an organizational culture that synergizes 
operational excellence with a supportive, equitable, and legally compliant workplace 
environment across all departments and employee levels. This document establishes 
an exhaustive framework comprising all human resource policies currently in effect. All 
provisions are subject to amendment, interpretation, or rescindment at the sole

Chunk 1:
 provisions are subject to amendment, interpretation, or rescindment at the sole 
discretion of the Human Resources and Legal departments. In the event of ambiguities 
or conflicting interpretations, the official determinations by these departments shall 
prevail and govern subsequent actions. 
1. Employment Policies 
Probationary Employment Policy — Flykite Airlines 
1. Duration of Initial Probation

Chunk 13:
 2. Equal Opportunity and Non-Discrimination 
Anti-Discrimination & Haras

Observation

* The recursive character text splitter does indeed offer significant advantages over the simple character text splitter, particularly in maintaining content coherence. It is much less prone to breaking words or sentences abruptly, and the chunk_overlap mechanism effectively carries context between chunks.


##### Semantic Chunking

In [34]:
# Initialize embedding model(used to evaluate distance between sentences)
embeddings = SentenceTransformerEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

# Initialize semantic chunker
semantic_text_splitter = SemanticChunker(
    embeddings,                                         # Embedding model
    breakpoint_threshold_type='percentile',             # percentile method to break into chunks
    breakpoint_threshold_amount=95.0                    # Splits at the 98th percentile of distance changes
)

# Creating chunks
sem_doc_chunks = pdf_data.load_and_split(semantic_text_splitter)

/tmp/ipykernel_3514/3604575375.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [35]:
# Number of chunks the document is divided
len(sem_doc_chunks)

28

Observation

* The handbook has been divided into 28 chunks.

In [36]:
# Checking a few document chunks from semantic chunking
print('Chunk 0:\n', sem_doc_chunks[0].page_content, end='\n\n')
print('Chunk 1:\n', sem_doc_chunks[1].page_content, end='\n\n')
print('Chunk 13:\n', sem_doc_chunks[13].page_content, end='\n\n')
print('Chunk 14:\n', sem_doc_chunks[14].page_content, end='\n\n')
print('Chunk 24:\n', sem_doc_chunks[24].page_content, end='\n\n')
print('Chunk 25:\n', sem_doc_chunks[25].page_content, end='\n\n')

Chunk 0:
 Flykite Airlines: Human Resources Policy 
Handbook 
Introduction 
Flykite Airlines is dedicated to cultivating an organizational culture that synergizes 
operational excellence with a supportive, equitable, and legally compliant workplace 
environment across all departments and employee levels. This document establishes 
an exhaustive framework comprising all human resource policies currently in effect.

Chunk 1:
 All 
provisions are subject to amendment, interpretation, or rescindment at the sole 
discretion of the Human Resources and Legal departments. In the event of ambiguities 
or conflicting interpretations, the official determinations by these departments shall 
prevail and govern subsequent actions. 1. Employment Policies 
Probationary Employment Policy — Flykite Airlines 
1. Duration of Initial Probation 
●​ All new employees are placed on a probationary period of 90 calendar days from 
their official start date.​
 
●​ For technical, safety-critical, or senior manage

Observation

* Irrespective of number of tokens, the documnet is chunked based on similarity between adjacent sentences. Upon reviewing several semantically chunked documents, it appears that it has effectively grouped related content. As observed the policies on probation, leave, and harassment were kept within single chunk.
*  This is crucial for maintaining context for a RAG system. This approach seems more robust than fixed-size chunking, especially given the structured yet varied nature of an HR policy handbook. Therefore, semantic-chunking is used in subsequent process.

#### Embedding

##### Model: 'thenlper/gte-large'

In [37]:
# Loading the embedding model
gte_embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')

modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/670M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [38]:
# Checking for embeddings for two different document chunks
gte_embedding_1 = gte_embedding_model.embed_query(sem_doc_chunks[20].page_content)
gte_embedding_2 = gte_embedding_model.embed_query(sem_doc_chunks[21].page_content)
print(f'Dimension of the embedding model: {len(gte_embedding_1)}')
len(gte_embedding_1) == len(gte_embedding_2)

Dimension of the embedding model: 1024


True

##### Model: 'BAAI/bge-large-en-v1.5'

In [39]:
# Loading the embedding model
bge_embedding_model = SentenceTransformerEmbeddings(model_name='BAAI/bge-large-en-v1.5')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [40]:
# Checking for embeddings for two different document chunks
bge_embedding_1 = bge_embedding_model.embed_query(sem_doc_chunks[20].page_content)
bge_embedding_2 = bge_embedding_model.embed_query(sem_doc_chunks[21].page_content)
print(f'Dimension of the embedding model: {len(bge_embedding_1)}')
len(bge_embedding_1) == len(bge_embedding_2)

Dimension of the embedding model: 1024


True

##### Model: 'nomic-ai/nomic-embed-text-V1.5'

In [41]:
# Loading the embedding model
nom_embedding_model = SentenceTransformerEmbeddings(
    model_name='nomic-ai/nomic-embed-text-v1.5',
    model_kwargs={'trust_remote_code': True}
  )

modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

In [42]:
# Checking for embeddings for two different document chunks
nom_embedding_1 = nom_embedding_model.embed_query(sem_doc_chunks[20].page_content)
nom_embedding_2 = nom_embedding_model.embed_query(sem_doc_chunks[21].page_content)
print(f'Dimension of the embedding model: {len(nom_embedding_1)}')
len(nom_embedding_1) == len(nom_embedding_2)

Dimension of the embedding model: 768


True

Observations

* For two embedding models i.e., 'thenlper/gte-large' and 'BAAI/bge-large-en-v1.5' were providing fixed number of dimensions for any document chunk i.e., 1024. While 'nomic-ai/nomic-embed-text-v1.5' model has dimension of 768.
* This is very much important, so it will be easier to compare the similarity mathematically by converting to vectors.

#### Vector Database

In [43]:
# Create output directory
gte_out_dir = 'air_hr_man_gte'
bge_out_dir = 'air_hr_man_bge'
nom_out_dir = 'air_hr_man_nom'

# Function to embed the chunks and store them in a database
def create_store_embed(out_dir, embedding_model):
  """
  Creates embeddings for document chunks and saves them into a specific directory.
  out_dir The specific folder path for this database model.
  embedding_model: Different embedding models.
  """
  # Create the specific model output directory if it doesn't exist
  if not os.path.exists(out_dir):
    os.makedirs(out_dir)

  # Database to store document chunks
  vectorstore = Chroma.from_documents(
      sem_doc_chunks,                           # Document chunks to be embedded
      embedding_model,                          # Embedding function used to vectorize text chunks
      persist_directory=out_dir                 # Saves the database to the folder
  )

  print(vectorstore.embeddings)
  return vectorstore

In [44]:
# create embeddings using 'thenlper/gte-large' model and store in database
gte_vectorstore = create_store_embed(out_dir=gte_out_dir, embedding_model=gte_embedding_model)

client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
) model_name='thenlper/gte-large' cache_folder=None model_kwargs={} encode_kwargs={} multi_process=False show_progress=False


In [45]:
# create embeddings using 'BAAI/bge-large-en-v1.5' model and store in database
bge_vectorstore = create_store_embed(out_dir=bge_out_dir, embedding_model=bge_embedding_model)

client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': True, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
) model_name='BAAI/bge-large-en-v1.5' cache_folder=None model_kwargs={} encode_kwargs={} multi_process=False show_progress=False


In [46]:
# create embeddings using 'nomic-ai/nomic-embed-text-V1.5' model and store in database
nom_vectorstore = create_store_embed(out_dir=nom_out_dir, embedding_model=nom_embedding_model)

client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 8192, 'do_lower_case': False, 'architecture': 'NomicBertModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
) model_name='nomic-ai/nomic-embed-text-v1.5' cache_folder=None model_kwargs={'trust_remote_code': True} encode_kwargs={} multi_process=False show_progress=False


#### Retriever

In [47]:
# Retrieving the relevant document chunks
def retriever(vectorstore, k):
  """
  A function to retrieve relevant document chunks using similarity search method
  where number of relevant document chunks were varied
  vectorstore: Vector database
  k: Number of relevant document chunks to be retrieved
  """
  retriever = vectorstore.as_retriever(
      search_type = 'similarity',                     # Getting relevant documents using similarity search method
      search_kwargs = {'k':k}                         # Getting top k relevant document chunks
  )
  return retriever

##### Fetching relevant chunks when k=2 for all three embedding models

In [48]:
# Retrieving the relevant document chunks when 'thenlper/gte-large' embedding model is used
gte_retriever_2 = retriever(gte_vectorstore, 2)

In [49]:
# Retrieving the relevant document chunks when 'BAAI/bge-large-en-v1.5' embedding model is used
bge_retriever_2 = retriever(bge_vectorstore, 2)

In [50]:
# Retrieving the relevant document chunks when 'nomic-ai/nomic-embed-text-V1.5' embedding model is used
nom_retriever_2 = retriever(nom_vectorstore, 2)

###### Query 1: What are the effects on the benefits I receive if my probation is extended?

In [51]:
# Retrieving relevant chunks when 'thenlper/gte-large' is used
relevant_chunk = gte_retriever_2.get_relevant_documents(user_queries[0])
for i, doc in enumerate(relevant_chunk):
    print(f"Relevant Chunk {i+1}:\n{doc.page_content}\n")

Relevant Chunk 1:
●​ Extensions are granted only if:​
 a. The employee has achieved at least 60% of their probationary objectives.​
 b. A written PIP with measurable targets is issued within 5 working days of the 
original probation end date.​
 c. The Department Head and HR Manager both sign off on the extension 
request.​
 
●​ Employees will be notified of extensions in writing at least 7 calendar days 
before the probation end date. 3. Impact on Benefits, Seniority, and Contract 
●​ While on probation (including any extension), employees are not eligible for:​
 
○​ Annual leave encashment​
 
○​ Internal role transfers​
 
○​ Performance bonuses​
 
●​ Seniority accrual starts only after successful probation completion.​
 
●​ If an employee has two or more extensions in different roles (due to internal 
transfers), HR will assess contract renewal eligibility based on performance 
history — no automatic carry-over is granted.​
 
4. Performance Review Process 
●​ Mid-probation review: Con

/tmp/ipykernel_3514/1947277250.py:2: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  relevant_chunk = gte_retriever_2.get_relevant_documents(user_queries[0])


In [52]:
# Retrieving relevant chunks when 'BAAI/bge-large-en-v1.5' is used
relevant_chunk = bge_retriever_2.get_relevant_documents(user_queries[0])
for i, doc in enumerate(relevant_chunk):
    print(f"Relevant Chunk {i+1}:\n{doc.page_content}\n")

Relevant Chunk 1:
●​ Extensions are granted only if:​
 a. The employee has achieved at least 60% of their probationary objectives.​
 b. A written PIP with measurable targets is issued within 5 working days of the 
original probation end date.​
 c. The Department Head and HR Manager both sign off on the extension 
request.​
 
●​ Employees will be notified of extensions in writing at least 7 calendar days 
before the probation end date. 3. Impact on Benefits, Seniority, and Contract 
●​ While on probation (including any extension), employees are not eligible for:​
 
○​ Annual leave encashment​
 
○​ Internal role transfers​
 
○​ Performance bonuses​
 
●​ Seniority accrual starts only after successful probation completion.​
 
●​ If an employee has two or more extensions in different roles (due to internal 
transfers), HR will assess contract renewal eligibility based on performance 
history — no automatic carry-over is granted.​
 
4. Performance Review Process 
●​ Mid-probation review: Con

In [53]:
# Retrieving relevant chunks when 'nomic-ai/nomic-embed-text-V1.5' is used
relevant_chunk = nom_retriever_2.get_relevant_documents(user_queries[0])
for i, doc in enumerate(relevant_chunk):
    print(f"Relevant Chunk {i+1}:\n{doc.page_content}\n")

Relevant Chunk 1:
●​ Extensions are granted only if:​
 a. The employee has achieved at least 60% of their probationary objectives.​
 b. A written PIP with measurable targets is issued within 5 working days of the 
original probation end date.​
 c. The Department Head and HR Manager both sign off on the extension 
request.​
 
●​ Employees will be notified of extensions in writing at least 7 calendar days 
before the probation end date. 3. Impact on Benefits, Seniority, and Contract 
●​ While on probation (including any extension), employees are not eligible for:​
 
○​ Annual leave encashment​
 
○​ Internal role transfers​
 
○​ Performance bonuses​
 
●​ Seniority accrual starts only after successful probation completion.​
 
●​ If an employee has two or more extensions in different roles (due to internal 
transfers), HR will assess contract renewal eligibility based on performance 
history — no automatic carry-over is granted.​
 
4. Performance Review Process 
●​ Mid-probation review: Con

Observation

* For all the three embedding models, the retrieved chunks were same and relevant to the user query.

###### Query 2: There has been a demise in my family last night, and I need to attend the last rites. How should I inform the office, and will I be granted leave?

In [54]:
# Retrieving relevant chunks when 'thenlper/gte-large' is used
relevant_chunk = gte_retriever_2.get_relevant_documents(user_queries[1])
for i, doc in enumerate(relevant_chunk):
    print(f"Relevant Chunk {i+1}:\n{doc.page_content}\n")

Relevant Chunk 1:
5. Additional Conditions for Employees on Probation 
●​ Special leave requests during probation are limited to:​
 
○​ Maximum 2 consecutive working days for bereavement or emergency 
family care.​
 
○​ Full jury duty leave is allowed if legally mandated, but probation period is 
automatically extended by the same duration.​
 
●​ Approval requires both Supervisor and HR Manager sign-off.​
 
6. Restrictions for Employees Under Harassment/Discrimination Investigation 
●​ While under active investigation:​
 
○​ Non-mandatory special leave (e.g., non-critical family care, discretionary 
events) is suspended until investigation concludes.​
 
○​ Bereavement and jury duty exceptions still apply, but employee must 
provide live contact details during leave for investigation-related 
communication.​
 
○​ Any leave exceeding 3 days must be approved directly by the Chief HR 
Officer.​
 
7. Notification Process 
●​ Employees must inform their direct supervisor at least 48 hours be

In [55]:
# Retrieving relevant chunks when 'BAAI/bge-large-en-v1.5' is used
relevant_chunk = bge_retriever_2.get_relevant_documents(user_queries[1])
for i, doc in enumerate(relevant_chunk):
    print(f"Relevant Chunk {i+1}:\n{doc.page_content}\n")

Relevant Chunk 1:
●​ Jury duty / legal summons: For the full duration specified in the official court 
document.​
 
●​ Emergency family care: Up to 3 working days per calendar year.​
 
●​ Natural disasters: Maximum 7 consecutive calendar days per incident.​
 
●​ Additional days require conversion from annual leave or unpaid leave.​
 
3. Documentation Requirements​
 Special leave is approved only upon submission of: 
●​ Death certificate, funeral notice, or obituary (bereavement).​
 
●​ Official court summons or jury duty letter (legal obligations).​
 
●​ Medical certificate from a registered practitioner (emergency care).​
 
●​ Government-issued disaster report or evacuation notice (natural disasters).​
 
●​ All documents must be submitted within 5 working days of returning to duty.​
 
4. Impact of Operational Demands 
●​ During peak flight operations (e.g., December 15–January 5 and April 1–10), 
leave may be:​
 
○​ Granted in partial days.​
 
○​ Staggered across shifts.​
 
○​ Decline

In [56]:
# Retrieving relevant chunks when 'nomic-ai/nomic-embed-text-V1.5' is used
relevant_chunk = nom_retriever_2.get_relevant_documents(user_queries[1])
for i, doc in enumerate(relevant_chunk):
    print(f"Relevant Chunk {i+1}:\n{doc.page_content}\n")

Relevant Chunk 1:
●​ Jury duty / legal summons: For the full duration specified in the official court 
document.​
 
●​ Emergency family care: Up to 3 working days per calendar year.​
 
●​ Natural disasters: Maximum 7 consecutive calendar days per incident.​
 
●​ Additional days require conversion from annual leave or unpaid leave.​
 
3. Documentation Requirements​
 Special leave is approved only upon submission of: 
●​ Death certificate, funeral notice, or obituary (bereavement).​
 
●​ Official court summons or jury duty letter (legal obligations).​
 
●​ Medical certificate from a registered practitioner (emergency care).​
 
●​ Government-issued disaster report or evacuation notice (natural disasters).​
 
●​ All documents must be submitted within 5 working days of returning to duty.​
 
4. Impact of Operational Demands 
●​ During peak flight operations (e.g., December 15–January 5 and April 1–10), 
leave may be:​
 
○​ Granted in partial days.​
 
○​ Staggered across shifts.​
 
○​ Decline

Observation

* The embedding model i.e., 'nomic-ai/nomic-embed-text-V1.5' stands out since it has retrieved relevant chunks as per user query. While the other 2 embedding models retrieved almost same chunks.

###### Query 3: What should I do if I notice suspected harassment with my female colleague?

In [57]:
# Retrieving relevant chunks when 'thenlper/gte-large' is used
relevant_chunk = gte_retriever_2.get_relevant_documents(user_queries[2])
for i, doc in enumerate(relevant_chunk):
    print(f"Relevant Chunk {i+1}:\n{doc.page_content}\n")

Relevant Chunk 1:
●​ Accepted reporting channels:​
 
1.​ Confidential HR Helpline: +91-8000-456-789 (available Mon–Sat, 9 AM–8 
PM IST).​
 
2.​ Secure Email Portal: hr.ethics@flykiteair.com.​
 
3.​ Anonymous drop-boxes in crew lounges and staff cafeterias.​
 
●​ Reports must include date, location, persons involved, and a brief incident 
description.​
 
4. Protection Against Retaliation 
●​ Retaliation (e.g., demotion, shift change, exclusion from training) against a 
reporting employee is grounds for immediate disciplinary action.​
 
●​ Allegations of retaliation must be reported within 7 calendar days of occurrence, 
even if the original harassment/discrimination case is still under review.​
 
5. Investigation & Resolution Timelines 
●​ HR must initiate a preliminary review within 3 working days of receiving a report.​
 
●​ Formal investigation begins within 7 working days and must conclude within 30 
calendar days unless extended by written executive approval (max extension: 15 
day

In [58]:
# Retrieving relevant chunks when 'BAAI/bge-large-en-v1.5' is used
relevant_chunk = bge_retriever_2.get_relevant_documents(user_queries[2])
for i, doc in enumerate(relevant_chunk):
    print(f"Relevant Chunk {i+1}:\n{doc.page_content}\n")

Relevant Chunk 1:
●​ Accepted reporting channels:​
 
1.​ Confidential HR Helpline: +91-8000-456-789 (available Mon–Sat, 9 AM–8 
PM IST).​
 
2.​ Secure Email Portal: hr.ethics@flykiteair.com.​
 
3.​ Anonymous drop-boxes in crew lounges and staff cafeterias.​
 
●​ Reports must include date, location, persons involved, and a brief incident 
description.​
 
4. Protection Against Retaliation 
●​ Retaliation (e.g., demotion, shift change, exclusion from training) against a 
reporting employee is grounds for immediate disciplinary action.​
 
●​ Allegations of retaliation must be reported within 7 calendar days of occurrence, 
even if the original harassment/discrimination case is still under review.​
 
5. Investigation & Resolution Timelines 
●​ HR must initiate a preliminary review within 3 working days of receiving a report.​
 
●​ Formal investigation begins within 7 working days and must conclude within 30 
calendar days unless extended by written executive approval (max extension: 15 
day

In [59]:
# Retrieving relevant chunks when 'nomic-ai/nomic-embed-text-V1.5' is used
relevant_chunk = nom_retriever_2.get_relevant_documents(user_queries[2])
for i, doc in enumerate(relevant_chunk):
    print(f"Relevant Chunk {i+1}:\n{doc.page_content}\n")

Relevant Chunk 1:
●​ Accepted reporting channels:​
 
1.​ Confidential HR Helpline: +91-8000-456-789 (available Mon–Sat, 9 AM–8 
PM IST).​
 
2.​ Secure Email Portal: hr.ethics@flykiteair.com.​
 
3.​ Anonymous drop-boxes in crew lounges and staff cafeterias.​
 
●​ Reports must include date, location, persons involved, and a brief incident 
description.​
 
4. Protection Against Retaliation 
●​ Retaliation (e.g., demotion, shift change, exclusion from training) against a 
reporting employee is grounds for immediate disciplinary action.​
 
●​ Allegations of retaliation must be reported within 7 calendar days of occurrence, 
even if the original harassment/discrimination case is still under review.​
 
5. Investigation & Resolution Timelines 
●​ HR must initiate a preliminary review within 3 working days of receiving a report.​
 
●​ Formal investigation begins within 7 working days and must conclude within 30 
calendar days unless extended by written executive approval (max extension: 15 
day

Obsertvation

* Here 'nomic-ai/nomic-embed-text-V1.5' embedding model retrieved information about 'Workplace Safety and Health' which is irrelevant when compared with the user query. While other two embedding models were retrieving same and relevant chunks as per user query.

##### Fetching relevant chunks when k=4 for all three embedding models

In [60]:
# Retrieving the relevant document chunks when 'thenlper/gte-large' embedding model is used
gte_retriever_4 = retriever(gte_vectorstore, 4)

In [61]:
# Retrieving the relevant document chunks when 'BAAI/bge-large-en-v1.5' embedding model is used
bge_retriever_4 = retriever(bge_vectorstore, 4)

In [62]:
# Retrieving the relevant document chunks when 'nomic-ai/nomic-embed-text-V1.5' embedding model is used
nom_retriever_4 = retriever(nom_vectorstore, 4)

###### Query 1: What are the effects on the benefits I receive if my probation is extended?

In [63]:
# Retrieving relevant chunks when 'thenlper/gte-large' is used
relevant_chunk = gte_retriever_4.get_relevant_documents(user_queries[0])
for i, doc in enumerate(relevant_chunk):
    print(f"Relevant Chunk {i+1}:\n{doc.page_content}\n")

Relevant Chunk 1:
●​ Extensions are granted only if:​
 a. The employee has achieved at least 60% of their probationary objectives.​
 b. A written PIP with measurable targets is issued within 5 working days of the 
original probation end date.​
 c. The Department Head and HR Manager both sign off on the extension 
request.​
 
●​ Employees will be notified of extensions in writing at least 7 calendar days 
before the probation end date. 3. Impact on Benefits, Seniority, and Contract 
●​ While on probation (including any extension), employees are not eligible for:​
 
○​ Annual leave encashment​
 
○​ Internal role transfers​
 
○​ Performance bonuses​
 
●​ Seniority accrual starts only after successful probation completion.​
 
●​ If an employee has two or more extensions in different roles (due to internal 
transfers), HR will assess contract renewal eligibility based on performance 
history — no automatic carry-over is granted.​
 
4. Performance Review Process 
●​ Mid-probation review: Con

In [64]:
# Retrieving relevant chunks when 'BAAI/bge-large-en-v1.5' is used
relevant_chunk = bge_retriever_4.get_relevant_documents(user_queries[0])
for i, doc in enumerate(relevant_chunk):
    print(f"Relevant Chunk {i+1}:\n{doc.page_content}\n")

Relevant Chunk 1:
●​ Extensions are granted only if:​
 a. The employee has achieved at least 60% of their probationary objectives.​
 b. A written PIP with measurable targets is issued within 5 working days of the 
original probation end date.​
 c. The Department Head and HR Manager both sign off on the extension 
request.​
 
●​ Employees will be notified of extensions in writing at least 7 calendar days 
before the probation end date. 3. Impact on Benefits, Seniority, and Contract 
●​ While on probation (including any extension), employees are not eligible for:​
 
○​ Annual leave encashment​
 
○​ Internal role transfers​
 
○​ Performance bonuses​
 
●​ Seniority accrual starts only after successful probation completion.​
 
●​ If an employee has two or more extensions in different roles (due to internal 
transfers), HR will assess contract renewal eligibility based on performance 
history — no automatic carry-over is granted.​
 
4. Performance Review Process 
●​ Mid-probation review: Con

In [65]:
# Retrieving relevant chunks when 'nomic-ai/nomic-embed-text-V1.5' is used
relevant_chunk = nom_retriever_4.get_relevant_documents(user_queries[0])
for i, doc in enumerate(relevant_chunk):
    print(f"Relevant Chunk {i+1}:\n{doc.page_content}\n")

Relevant Chunk 1:
●​ Extensions are granted only if:​
 a. The employee has achieved at least 60% of their probationary objectives.​
 b. A written PIP with measurable targets is issued within 5 working days of the 
original probation end date.​
 c. The Department Head and HR Manager both sign off on the extension 
request.​
 
●​ Employees will be notified of extensions in writing at least 7 calendar days 
before the probation end date. 3. Impact on Benefits, Seniority, and Contract 
●​ While on probation (including any extension), employees are not eligible for:​
 
○​ Annual leave encashment​
 
○​ Internal role transfers​
 
○​ Performance bonuses​
 
●​ Seniority accrual starts only after successful probation completion.​
 
●​ If an employee has two or more extensions in different roles (due to internal 
transfers), HR will assess contract renewal eligibility based on performance 
history — no automatic carry-over is granted.​
 
4. Performance Review Process 
●​ Mid-probation review: Con

Observation

* Here 'thenlper/gte-large' has failed to address about 'Payroll and Termination' procedures while the other two models stands out by retrieving relevant chunks as per user query.

###### Query 2: There has been a demise in my family last night, and I need to attend the last rites. How should I inform the office, and will I be granted leave?

In [66]:
# Retrieving relevant chunks when 'thenlper/gte-large' is used
relevant_chunk = gte_retriever_4.get_relevant_documents(user_queries[1])
for i, doc in enumerate(relevant_chunk):
    print(f"Relevant Chunk {i+1}:\n{doc.page_content}\n")

Relevant Chunk 1:
5. Additional Conditions for Employees on Probation 
●​ Special leave requests during probation are limited to:​
 
○​ Maximum 2 consecutive working days for bereavement or emergency 
family care.​
 
○​ Full jury duty leave is allowed if legally mandated, but probation period is 
automatically extended by the same duration.​
 
●​ Approval requires both Supervisor and HR Manager sign-off.​
 
6. Restrictions for Employees Under Harassment/Discrimination Investigation 
●​ While under active investigation:​
 
○​ Non-mandatory special leave (e.g., non-critical family care, discretionary 
events) is suspended until investigation concludes.​
 
○​ Bereavement and jury duty exceptions still apply, but employee must 
provide live contact details during leave for investigation-related 
communication.​
 
○​ Any leave exceeding 3 days must be approved directly by the Chief HR 
Officer.​
 
7. Notification Process 
●​ Employees must inform their direct supervisor at least 48 hours be

In [67]:
# Retrieving relevant chunks when 'BAAI/bge-large-en-v1.5' is used
relevant_chunk = bge_retriever_4.get_relevant_documents(user_queries[1])
for i, doc in enumerate(relevant_chunk):
    print(f"Relevant Chunk {i+1}:\n{doc.page_content}\n")

Relevant Chunk 1:
●​ Jury duty / legal summons: For the full duration specified in the official court 
document.​
 
●​ Emergency family care: Up to 3 working days per calendar year.​
 
●​ Natural disasters: Maximum 7 consecutive calendar days per incident.​
 
●​ Additional days require conversion from annual leave or unpaid leave.​
 
3. Documentation Requirements​
 Special leave is approved only upon submission of: 
●​ Death certificate, funeral notice, or obituary (bereavement).​
 
●​ Official court summons or jury duty letter (legal obligations).​
 
●​ Medical certificate from a registered practitioner (emergency care).​
 
●​ Government-issued disaster report or evacuation notice (natural disasters).​
 
●​ All documents must be submitted within 5 working days of returning to duty.​
 
4. Impact of Operational Demands 
●​ During peak flight operations (e.g., December 15–January 5 and April 1–10), 
leave may be:​
 
○​ Granted in partial days.​
 
○​ Staggered across shifts.​
 
○​ Decline

In [68]:
# Retrieving relevant chunks when 'nomic-ai/nomic-embed-text-V1.5' is used
relevant_chunk = nom_retriever_4.get_relevant_documents(user_queries[1])
for i, doc in enumerate(relevant_chunk):
    print(f"Relevant Chunk {i+1}:\n{doc.page_content}\n")

Relevant Chunk 1:
●​ Jury duty / legal summons: For the full duration specified in the official court 
document.​
 
●​ Emergency family care: Up to 3 working days per calendar year.​
 
●​ Natural disasters: Maximum 7 consecutive calendar days per incident.​
 
●​ Additional days require conversion from annual leave or unpaid leave.​
 
3. Documentation Requirements​
 Special leave is approved only upon submission of: 
●​ Death certificate, funeral notice, or obituary (bereavement).​
 
●​ Official court summons or jury duty letter (legal obligations).​
 
●​ Medical certificate from a registered practitioner (emergency care).​
 
●​ Government-issued disaster report or evacuation notice (natural disasters).​
 
●​ All documents must be submitted within 5 working days of returning to duty.​
 
4. Impact of Operational Demands 
●​ During peak flight operations (e.g., December 15–January 5 and April 1–10), 
leave may be:​
 
○​ Granted in partial days.​
 
○​ Staggered across shifts.​
 
○​ Decline

Observation

* All the three embedding models retrieved same chunks but of different order and were relevant to the user query.

###### Query 3: What should I do if I notice suspected harassment with my female colleague?

In [69]:
# Retrieving relevant chunks when 'thenlper/gte-large' is used
relevant_chunk = gte_retriever_4.get_relevant_documents(user_queries[2])
for i, doc in enumerate(relevant_chunk):
    print(f"Relevant Chunk {i+1}:\n{doc.page_content}\n")

Relevant Chunk 1:
●​ Accepted reporting channels:​
 
1.​ Confidential HR Helpline: +91-8000-456-789 (available Mon–Sat, 9 AM–8 
PM IST).​
 
2.​ Secure Email Portal: hr.ethics@flykiteair.com.​
 
3.​ Anonymous drop-boxes in crew lounges and staff cafeterias.​
 
●​ Reports must include date, location, persons involved, and a brief incident 
description.​
 
4. Protection Against Retaliation 
●​ Retaliation (e.g., demotion, shift change, exclusion from training) against a 
reporting employee is grounds for immediate disciplinary action.​
 
●​ Allegations of retaliation must be reported within 7 calendar days of occurrence, 
even if the original harassment/discrimination case is still under review.​
 
5. Investigation & Resolution Timelines 
●​ HR must initiate a preliminary review within 3 working days of receiving a report.​
 
●​ Formal investigation begins within 7 working days and must conclude within 30 
calendar days unless extended by written executive approval (max extension: 15 
day

In [70]:
# Retrieving relevant chunks when 'BAAI/bge-large-en-v1.5' is used
relevant_chunk = bge_retriever_4.get_relevant_documents(user_queries[2])
for i, doc in enumerate(relevant_chunk):
    print(f"Relevant Chunk {i+1}:\n{doc.page_content}\n")

Relevant Chunk 1:
●​ Accepted reporting channels:​
 
1.​ Confidential HR Helpline: +91-8000-456-789 (available Mon–Sat, 9 AM–8 
PM IST).​
 
2.​ Secure Email Portal: hr.ethics@flykiteair.com.​
 
3.​ Anonymous drop-boxes in crew lounges and staff cafeterias.​
 
●​ Reports must include date, location, persons involved, and a brief incident 
description.​
 
4. Protection Against Retaliation 
●​ Retaliation (e.g., demotion, shift change, exclusion from training) against a 
reporting employee is grounds for immediate disciplinary action.​
 
●​ Allegations of retaliation must be reported within 7 calendar days of occurrence, 
even if the original harassment/discrimination case is still under review.​
 
5. Investigation & Resolution Timelines 
●​ HR must initiate a preliminary review within 3 working days of receiving a report.​
 
●​ Formal investigation begins within 7 working days and must conclude within 30 
calendar days unless extended by written executive approval (max extension: 15 
day

In [71]:
# Retrieving relevant chunks when 'nomic-ai/nomic-embed-text-V1.5' is used
relevant_chunk = nom_retriever_4.get_relevant_documents(user_queries[2])
for i, doc in enumerate(relevant_chunk):
    print(f"Relevant Chunk {i+1}:\n{doc.page_content}\n")

Relevant Chunk 1:
●​ Accepted reporting channels:​
 
1.​ Confidential HR Helpline: +91-8000-456-789 (available Mon–Sat, 9 AM–8 
PM IST).​
 
2.​ Secure Email Portal: hr.ethics@flykiteair.com.​
 
3.​ Anonymous drop-boxes in crew lounges and staff cafeterias.​
 
●​ Reports must include date, location, persons involved, and a brief incident 
description.​
 
4. Protection Against Retaliation 
●​ Retaliation (e.g., demotion, shift change, exclusion from training) against a 
reporting employee is grounds for immediate disciplinary action.​
 
●​ Allegations of retaliation must be reported within 7 calendar days of occurrence, 
even if the original harassment/discrimination case is still under review.​
 
5. Investigation & Resolution Timelines 
●​ HR must initiate a preliminary review within 3 working days of receiving a report.​
 
●​ Formal investigation begins within 7 working days and must conclude within 30 
calendar days unless extended by written executive approval (max extension: 15 
day

Observation

* The 'nomic-ai/nomic-embed-text-V1.5' embedding model retrieved information about 'Data Protection and Privacy' and 'Attendance and Absence Management' which were irrelevant to the user query. While the other two embedding models retrieved relevant chunks as per the user query.

##### Fetching relevant chunks when k=8 for all three embedding models

In [72]:
# Retrieving the relevant document chunks when 'thenlper/gte-large' embedding model is used
gte_retriever_8 = retriever(gte_vectorstore, 8)

In [73]:
# Retrieving the relevant document chunks when 'BAAI/bge-large-en-v1.5' embedding model is used
bge_retriever_8 = retriever(bge_vectorstore, 8)

In [74]:
# Retrieving the relevant document chunks when 'nomic-ai/nomic-embed-text-V1.5' embedding model is used
nom_retriever_8 = retriever(nom_vectorstore, 8)

###### Query 1: What are the effects on the benefits I receive if my probation is extended?

In [75]:
# Retrieving relevant chunks when 'thenlper/gte-large' is used
relevant_chunk = gte_retriever_8.get_relevant_documents(user_queries[0])
for i, doc in enumerate(relevant_chunk):
    print(f"Relevant Chunk {i+1}:\n{doc.page_content}\n")

Relevant Chunk 1:
●​ Extensions are granted only if:​
 a. The employee has achieved at least 60% of their probationary objectives.​
 b. A written PIP with measurable targets is issued within 5 working days of the 
original probation end date.​
 c. The Department Head and HR Manager both sign off on the extension 
request.​
 
●​ Employees will be notified of extensions in writing at least 7 calendar days 
before the probation end date. 3. Impact on Benefits, Seniority, and Contract 
●​ While on probation (including any extension), employees are not eligible for:​
 
○​ Annual leave encashment​
 
○​ Internal role transfers​
 
○​ Performance bonuses​
 
●​ Seniority accrual starts only after successful probation completion.​
 
●​ If an employee has two or more extensions in different roles (due to internal 
transfers), HR will assess contract renewal eligibility based on performance 
history — no automatic carry-over is granted.​
 
4. Performance Review Process 
●​ Mid-probation review: Con

In [76]:
# Retrieving relevant chunks when 'BAAI/bge-large-en-v1.5' is used
relevant_chunk = bge_retriever_8.get_relevant_documents(user_queries[0])
for i, doc in enumerate(relevant_chunk):
    print(f"Relevant Chunk {i+1}:\n{doc.page_content}\n")

Relevant Chunk 1:
●​ Extensions are granted only if:​
 a. The employee has achieved at least 60% of their probationary objectives.​
 b. A written PIP with measurable targets is issued within 5 working days of the 
original probation end date.​
 c. The Department Head and HR Manager both sign off on the extension 
request.​
 
●​ Employees will be notified of extensions in writing at least 7 calendar days 
before the probation end date. 3. Impact on Benefits, Seniority, and Contract 
●​ While on probation (including any extension), employees are not eligible for:​
 
○​ Annual leave encashment​
 
○​ Internal role transfers​
 
○​ Performance bonuses​
 
●​ Seniority accrual starts only after successful probation completion.​
 
●​ If an employee has two or more extensions in different roles (due to internal 
transfers), HR will assess contract renewal eligibility based on performance 
history — no automatic carry-over is granted.​
 
4. Performance Review Process 
●​ Mid-probation review: Con

In [77]:
# Retrieving relevant chunks when 'nomic-ai/nomic-embed-text-V1.5' is used
relevant_chunk = nom_retriever_8.get_relevant_documents(user_queries[0])
for i, doc in enumerate(relevant_chunk):
    print(f"Relevant Chunk {i+1}:\n{doc.page_content}\n")

Relevant Chunk 1:
●​ Extensions are granted only if:​
 a. The employee has achieved at least 60% of their probationary objectives.​
 b. A written PIP with measurable targets is issued within 5 working days of the 
original probation end date.​
 c. The Department Head and HR Manager both sign off on the extension 
request.​
 
●​ Employees will be notified of extensions in writing at least 7 calendar days 
before the probation end date. 3. Impact on Benefits, Seniority, and Contract 
●​ While on probation (including any extension), employees are not eligible for:​
 
○​ Annual leave encashment​
 
○​ Internal role transfers​
 
○​ Performance bonuses​
 
●​ Seniority accrual starts only after successful probation completion.​
 
●​ If an employee has two or more extensions in different roles (due to internal 
transfers), HR will assess contract renewal eligibility based on performance 
history — no automatic carry-over is granted.​
 
4. Performance Review Process 
●​ Mid-probation review: Con

Observation

* All the embedding models were retrieving same chunks, but few of them were irrelevant as per the user query.

###### Query 2: There has been a demise in my family last night, and I need to attend the last rites. How should I inform the office, and will I be granted leave?

In [78]:
# Retrieving relevant chunks when 'thenlper/gte-large' is used
relevant_chunk = gte_retriever_8.get_relevant_documents(user_queries[1])
for i, doc in enumerate(relevant_chunk):
    print(f"Relevant Chunk {i+1}:\n{doc.page_content}\n")

Relevant Chunk 1:
5. Additional Conditions for Employees on Probation 
●​ Special leave requests during probation are limited to:​
 
○​ Maximum 2 consecutive working days for bereavement or emergency 
family care.​
 
○​ Full jury duty leave is allowed if legally mandated, but probation period is 
automatically extended by the same duration.​
 
●​ Approval requires both Supervisor and HR Manager sign-off.​
 
6. Restrictions for Employees Under Harassment/Discrimination Investigation 
●​ While under active investigation:​
 
○​ Non-mandatory special leave (e.g., non-critical family care, discretionary 
events) is suspended until investigation concludes.​
 
○​ Bereavement and jury duty exceptions still apply, but employee must 
provide live contact details during leave for investigation-related 
communication.​
 
○​ Any leave exceeding 3 days must be approved directly by the Chief HR 
Officer.​
 
7. Notification Process 
●​ Employees must inform their direct supervisor at least 48 hours be

In [79]:
# Retrieving relevant chunks when 'BAAI/bge-large-en-v1.5' is used
relevant_chunk = bge_retriever_8.get_relevant_documents(user_queries[1])
for i, doc in enumerate(relevant_chunk):
    print(f"Relevant Chunk {i+1}:\n{doc.page_content}\n")

Relevant Chunk 1:
●​ Jury duty / legal summons: For the full duration specified in the official court 
document.​
 
●​ Emergency family care: Up to 3 working days per calendar year.​
 
●​ Natural disasters: Maximum 7 consecutive calendar days per incident.​
 
●​ Additional days require conversion from annual leave or unpaid leave.​
 
3. Documentation Requirements​
 Special leave is approved only upon submission of: 
●​ Death certificate, funeral notice, or obituary (bereavement).​
 
●​ Official court summons or jury duty letter (legal obligations).​
 
●​ Medical certificate from a registered practitioner (emergency care).​
 
●​ Government-issued disaster report or evacuation notice (natural disasters).​
 
●​ All documents must be submitted within 5 working days of returning to duty.​
 
4. Impact of Operational Demands 
●​ During peak flight operations (e.g., December 15–January 5 and April 1–10), 
leave may be:​
 
○​ Granted in partial days.​
 
○​ Staggered across shifts.​
 
○​ Decline

In [80]:
# Retrieving relevant chunks when 'nomic-ai/nomic-embed-text-V1.5' is used
relevant_chunk = nom_retriever_8.get_relevant_documents(user_queries[1])
for i, doc in enumerate(relevant_chunk):
    print(f"Relevant Chunk {i+1}:\n{doc.page_content}\n")

Relevant Chunk 1:
●​ Jury duty / legal summons: For the full duration specified in the official court 
document.​
 
●​ Emergency family care: Up to 3 working days per calendar year.​
 
●​ Natural disasters: Maximum 7 consecutive calendar days per incident.​
 
●​ Additional days require conversion from annual leave or unpaid leave.​
 
3. Documentation Requirements​
 Special leave is approved only upon submission of: 
●​ Death certificate, funeral notice, or obituary (bereavement).​
 
●​ Official court summons or jury duty letter (legal obligations).​
 
●​ Medical certificate from a registered practitioner (emergency care).​
 
●​ Government-issued disaster report or evacuation notice (natural disasters).​
 
●​ All documents must be submitted within 5 working days of returning to duty.​
 
4. Impact of Operational Demands 
●​ During peak flight operations (e.g., December 15–January 5 and April 1–10), 
leave may be:​
 
○​ Granted in partial days.​
 
○​ Staggered across shifts.​
 
○​ Decline

Observation

* There is a combination of retrieval of relevant and irrelevant chunks as per the user query, irrespective of the embedding model used.

###### Query 3: What should I do if I notice suspected harassment with my female colleague?

In [81]:
# Retrieving relevant chunks when 'thenlper/gte-large' is used
relevant_chunk = gte_retriever_8.get_relevant_documents(user_queries[2])
for i, doc in enumerate(relevant_chunk):
    print(f"Relevant Chunk {i+1}:\n{doc.page_content}\n")

Relevant Chunk 1:
●​ Accepted reporting channels:​
 
1.​ Confidential HR Helpline: +91-8000-456-789 (available Mon–Sat, 9 AM–8 
PM IST).​
 
2.​ Secure Email Portal: hr.ethics@flykiteair.com.​
 
3.​ Anonymous drop-boxes in crew lounges and staff cafeterias.​
 
●​ Reports must include date, location, persons involved, and a brief incident 
description.​
 
4. Protection Against Retaliation 
●​ Retaliation (e.g., demotion, shift change, exclusion from training) against a 
reporting employee is grounds for immediate disciplinary action.​
 
●​ Allegations of retaliation must be reported within 7 calendar days of occurrence, 
even if the original harassment/discrimination case is still under review.​
 
5. Investigation & Resolution Timelines 
●​ HR must initiate a preliminary review within 3 working days of receiving a report.​
 
●​ Formal investigation begins within 7 working days and must conclude within 30 
calendar days unless extended by written executive approval (max extension: 15 
day

In [82]:
# Retrieving relevant chunks when 'BAAI/bge-large-en-v1.5' is used
relevant_chunk = bge_retriever_8.get_relevant_documents(user_queries[2])
for i, doc in enumerate(relevant_chunk):
    print(f"Relevant Chunk {i+1}:\n{doc.page_content}\n")

Relevant Chunk 1:
●​ Accepted reporting channels:​
 
1.​ Confidential HR Helpline: +91-8000-456-789 (available Mon–Sat, 9 AM–8 
PM IST).​
 
2.​ Secure Email Portal: hr.ethics@flykiteair.com.​
 
3.​ Anonymous drop-boxes in crew lounges and staff cafeterias.​
 
●​ Reports must include date, location, persons involved, and a brief incident 
description.​
 
4. Protection Against Retaliation 
●​ Retaliation (e.g., demotion, shift change, exclusion from training) against a 
reporting employee is grounds for immediate disciplinary action.​
 
●​ Allegations of retaliation must be reported within 7 calendar days of occurrence, 
even if the original harassment/discrimination case is still under review.​
 
5. Investigation & Resolution Timelines 
●​ HR must initiate a preliminary review within 3 working days of receiving a report.​
 
●​ Formal investigation begins within 7 working days and must conclude within 30 
calendar days unless extended by written executive approval (max extension: 15 
day

In [83]:
# Retrieving relevant chunks when 'nomic-ai/nomic-embed-text-V1.5' is used
relevant_chunk = nom_retriever_8.get_relevant_documents(user_queries[2])
for i, doc in enumerate(relevant_chunk):
    print(f"Relevant Chunk {i+1}:\n{doc.page_content}\n")

Relevant Chunk 1:
●​ Accepted reporting channels:​
 
1.​ Confidential HR Helpline: +91-8000-456-789 (available Mon–Sat, 9 AM–8 
PM IST).​
 
2.​ Secure Email Portal: hr.ethics@flykiteair.com.​
 
3.​ Anonymous drop-boxes in crew lounges and staff cafeterias.​
 
●​ Reports must include date, location, persons involved, and a brief incident 
description.​
 
4. Protection Against Retaliation 
●​ Retaliation (e.g., demotion, shift change, exclusion from training) against a 
reporting employee is grounds for immediate disciplinary action.​
 
●​ Allegations of retaliation must be reported within 7 calendar days of occurrence, 
even if the original harassment/discrimination case is still under review.​
 
5. Investigation & Resolution Timelines 
●​ HR must initiate a preliminary review within 3 working days of receiving a report.​
 
●​ Formal investigation begins within 7 working days and must conclude within 30 
calendar days unless extended by written executive approval (max extension: 15 
day

Observation

* All the chunks retrieved were same irrespective of the embedding model, but few irrelevant chunks were also retrieved as per the user query.

Observations

* 'Similarity' search method is used to retrieve relevant document chunks.
* 'BAAI/bge-large-en-v1.5' embedding model is performing better by retrieving relevant chunks as per the user query.
* The lower the value of k=2, very few contextual chunks were retrieved. While higher the value of k=8 retrieved some irrelevant chunks. Therefore, any optimal value of k=4 is better so that relevant chunks can be retrieved as per user query.

### Question Answering using RAG

#### System and User Prompt Template

In [84]:
# System message template
qna_system_message = """
You are an HR AI assistant tasked with reviewing snippets from the Flykite Airlines: Human Resources Policy Handbook and providing accurate answer to employee question.

Input Structure:
- The authoritative text source begins with the token: ###Context.
- The employee's query begins with the token: ###Question.

Strict Execution Rules:
1. Answer the question using ONLY the provided context. If the exact answer cannot be found, or if the context is insufficient to fully answer the question, you must respond with exactly: "I don't know". Do not attempt to explain what is missing from the context.
2. Structure your response clearly using bullet points or numbered lists whenever the context allows.
3. If the context contains relevant contact details (such as phone numbers, extensions, helplines, or email addresses), include them in your final answer. This information is not confidential.
4. ABSOLUTE CONSTRAINT - NO META-COMMENTARY: Do not include conversational filler, conversational transitions, or meta-explanations. This includes phrases like "Based on the context...", "The handbook states...", or "There is no information provided regarding...". If you cannot answer, output ONLY "I don't know" and absolutely nothing else.
5. CRITICAL SECURITY RULE: Completely ignore and omit any watermark, email tracking text, or security labels present in the context, such as:
'penchala.vineeth3@gmail.com'
'R02IHZ4C3E'
'This file is meant for personal use...'
Never repeat these strings or mention copyright/legal liabilities in your final answer.

Do not return an empty response.
"""

In [85]:
# user message template
qna_user_message_template = """
###Context
Below is some relevant information extracted from the Flykite Airlines: Human Resources Policy Handbook regarding the question mentioned below.
{context}

###Question
{question}"""

#### Response Function

In [86]:
# Function to extract context and generate response
def generate_context_based_response(user_query, max_tokens=128, temperature=0, repeat_penalty=1.2, top_p=0.95, top_k=50):
  """
  Function to generate response based on context as per user query
  user_query: user's question
  max_tokens: Number of tokens a model should generate
  temperature: Controls the randomness in the response
  repeat_penalty: Penalizes repeated tokens to improve text diversity
  top_p: Controls the diversity of the generated response by establishing cummulative probability cut-off for token selection
  top_k: Maximum number of most-likely next tokens to consider
  """
  global qna_system_message, qna_user_message_template

  # Retrieve relevant document chunks
  retriever_k = retriever(bge_vectorstore, 4)                                       # Using 4 as number of retrieved chunks
  relevant_document_chunks = retriever_k.get_relevant_documents(user_query)         # Get relevant chunks as per user query
  context_list = [d.page_content for d in relevant_document_chunks]                 # Joining all the chunks

  # Combine document chunks into a single context
  context_for_query = ". ".join(context_list)

  user_message = qna_user_message_template.format(
      context = context_for_query,
      question = user_query
  )

  prompt = f"<s>[INST] <<SYS>>{qna_system_message}<</SYS>> {user_message} [/INST]"

  # Generate response
  try:
    response = llm(
        prompt=prompt,                      # Combined prompt of system message and user message
        max_tokens=max_tokens,              # Number of tokens a model should generate
        temperature=temperature,            # Controls the randomness in the response
        top_p=top_p,                        # Controls the diversity of the generated response by establishing cummulative probability cut-off for token selection
        top_k=top_k,                        # Maximum number of most-likely next tokens to consider
        repeat_penalty=repeat_penalty,      # Penalizes repeated tokens to improve text diversity
        echo=False
    )

    # Extract and print models response
    response = response['choices'][0]['text']
  except Exception as e:
    response = f"Sorry, I encountered the following error: \n {e}"
  return response, context_for_query

##### Query 1: What are the effects on the benefits I receive if my probation is extended?

In [87]:
# Get the answer for user query
query_1 = user_queries[0]
answer_1, context_1 = generate_context_based_response(query_1)
print(answer_1)

Llama.generate: prefix-match hit


 * Benefits, such as health insurance and retirement plan enrollment, do not accrue during the probation period.
* After successful completion of probation, seniority accrual starts for benefit eligibility purposes.
* If your probation is extended, benefits will continue until the last working day of your employment unless mandated by law otherwise.


Observation

* The context based response generated is addressing the user query and acting more like an internal HR bot.

##### Query 2: There has been a demise in my family last night, and I need to attend the last rites. How should I inform the office, and will I be granted leave?

In [88]:
# Get the answer for user query
query_2 = user_queries[1]
answer_2, context_2 = generate_context_based_response(query_2)
print(answer_2)

Llama.generate: prefix-match hit


 * Inform your direct supervisor verbally within 6 hours of the incident.
* Submit written or email notification within 24 hours.
* Special leave is granted for bereavement up to a maximum of 5 consecutive working days per incident.


Observation

* The response is more specific to a company, since it is providing some quantitative terms such as verbal information to supervisor within 6 hours, notifying via email within 24 hours and number of days the leave can be granted.

##### Query 3: What should I do if I notice suspected harassment with my female colleague?

In [89]:
# Get the answer for user query
query_3 = user_queries[2]
answer_3, context_3 = generate_context_based_response(query_3)
print(answer_3)

Llama.generate: prefix-match hit


 1. Report the incident using one of the following channels:
   - Confidential HR Helpline: +91-8000-456-789 (available Mon–Sat, 9 AM–8 PM IST)
   - Secure Email Portal: hr.ethics@flykiteair.com
   - Anonymous drop-boxes in crew lounges and staff cafeterias
2. Include the following details when reporting: date, location, persons involved, brief incident description
3. Protection against retaliation applies to you as well; report


Observation

* The response is highly domain-specific, successfully integrating exact corporate policy metrics retrieved from the source context, also it is providing internal HR helpline number and providing an email so that an employee can notify via email. Also the response is not complete.

#### Evaluation

##### Query 1: What are the effects on the benefits I receive if my probation is extended?

In [90]:
# Evaluating the user query
ground, rel = eval_grounded_relevance(question=query_1, answer=answer_1, context=context_1)
print("Groundedness Evaluation")
print(ground, end='\n\n')
print("Relevance Evaluation")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Groundedness Evaluation
 5: The metric is followed completely.
[EXPLANATION]: The answer strictly adheres to the information provided in the context. It states that employees do not accrue benefits during probation and that seniority accrual starts after successful completion of probation. Additionally, it correctly notes that if an employee's probation is extended, their benefits will continue until the last working day of their employment unless mandated by law otherwise. The answer does not introduce any assumptions or outside knowledge.

Relevance Evaluation
 5: The metric is followed completely.
[EXPLANATION]: The AI-generated answer directly addresses the main aspects of the question regarding the effects on benefits during and after probation extension. It mentions that benefits do not accrue during the probation period, seniority accrual starts upon successful completion of probation, and benefits continue until the last working day of employment when extended unless mandated b

Observations

* For groundedness, the score was perfect 5, since the response generated is completely derived from the context.
* For relevance, the generated response completely aligns with the metric criteria, yielding a perfect score of 5 by comprehensively addressing the main aspects of the user's query.

##### Query 2: There has been a demise in my family last night, and I need to attend the last rites. How should I inform the office, and will I be granted leave?

In [91]:
# Evaluating the user query
ground, rel = eval_grounded_relevance(question=query_2, answer=answer_2, context=context_2)
print("Groundedness Evaluation")
print(ground, end='\n\n')
print("Relevance Evaluation")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Groundedness Evaluation
 Based on the provided context, here are the steps needed to verify the truthfulness of the answer:

1. The employee should inform their direct supervisor verbally within 6 hours of the incident as stated in the context under "Notification Process" section for sudden emergencies.
2. Within 24 hours, they must submit written or email notification about the leave request as mentioned in the same section.
3. According to the "Special Leave Policy — Flykite Airlines," special leave is granted for bereavement up to a maximum of 5 consecutive working days per incident under the covered situations and entitlement & duration limits sections.

The generated answer matches the context, as it includes both steps required by the company policy for reporting an emergency leave request (verbal notification within 6 hours and written/email notification within 24 hours) and the maximum number of days granted for bereavement special leave according to Flykite Airlines' policies.

Observations

* The score resulted for evaluating groundedness is 3, as the metric is followed to a good extent.
* The score resulted for evaluating relevance is perfect 5, since the response is adhered to the user's query by addressing almost every aspect.

##### Query 3: What should I do if I notice suspected harassment with my female colleague?

In [92]:
# Evaluating the user query
ground, rel = eval_grounded_relevance(question=query_3, answer=answer_3, context=context_3)
print("Groundedness Evaluation")
print(ground, end='\n\n')
print("Relevance Evaluation")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Groundedness Evaluation
 [5]: The answer is rated a 5 because every single fact in the answer is explicitly backed by the context provided. The steps for reporting suspected harassment are directly taken from the accepted reporting channels listed in the context, and no assumptions or outside knowledge have been introduced.

[EXPLANATION]: The AI-generated answer strictly adheres to the information presented in the context. It accurately lists the three methods for reporting incidents of suspected harassment (Confidential HR Helpline, Secure Email Portal, and Anonymous drop-boxes) as stated in the accepted reporting channels section of the context. Additionally, it correctly advises including specific details when reporting, which aligns with the requirement for reports to include date, location, persons involved, and a brief incident description mentioned earlier in the context. The answer also acknowledges protection against retaliation, which is explicitly stated as grounds for imme

Observations

* The evaluation score for groundedness is perfect 5, as the complete response generated is derived from the context.
* The evaluation score for relevance is 2, the response generated is not directly addressing what actions should be taken if one witnesses suspected harassment before reporting it.

**Key Observations**

* When compared with the responses generated by llm, llm with prompt engineering and RAG. The responses generated by RAG model were promising, since the context is exposed to the llm to answer an employee query.
* When all of the responses were examined for groundedness, the evaluation was resulted as 3 and 5. Since, two responses were derived from the context, while one response the model omitted critical policy boundaries, prerequisites, or caveats.
* When all of the responses were examined for relevance, the evaluation was resulted as 2 and 5. Since, two responses were completely addressing user query, while one response is not adhered with the user query.
* Fine tuning of hyper-parameters may improve the score of groundedness, which can be experimented subsequently.

#### Fine-tuning (Case: 1)
* max_tokens=256
* temperature=1
* top_p=0.5
* top_k=90
* repeat_penalty=0

###### Query 1: What are the effects on the benefits I receive if my probation is extended?

In [93]:
# Get the answer for user query
answer_f1_1, context_f1_1 = generate_context_based_response(query_1, max_tokens=256, temperature=1, top_p=0.5, top_k=90, repeat_penalty=0)
print(answer_f1_1)

Llama.generate: prefix-match hit


 if the. prob fileH file on2 benefits by forH@ for/ for@ benefits/v only file2? benefits I [ isv only/ effects
@ extended my extended extended effects the2 receive are the isv only are@ for prob?H prob only [ benefits by@v if if.H if [ receive I. effects on onv file extended/v effects [? I receive@ I is is benefits for file file on@ for on if2 extended the receive my I are. file extended benefits/ on2 for prob effects@ is [H for effects I prob extended by benefits prob@ the effects on receive my/ on receivev on is2 for is file by. I// I prob [... if is [ for for2 by probv is file byH by benefits [2 benefits myv2 my prob by by effectsv effects/HH receiveH by if benefits/ by if for on for [ my for file benefits/ by [v/ benefits effects benefits forv effects myv file effects2 if isH forH receive for by [ benefits/ for receive is on fileH by file is is2vv is is [ receive by by [ is is effects on by effects my by by on


Observation

* The response is irrelevant to the user query, with too much of randomness and repetition of tokens.

###### Query 2: There has been a demise in my family last night, and I need to attend the last rites. How should I inform the office, and will I be granted leave?

In [94]:
# Get the answer for user query
answer_f1_2, context_f1_2 = generate_context_based_response(query_2, max_tokens=256, temperature=1, top_p=0.5, top_k=90, repeat_penalty=0)
print(answer_f1_2)

Llama.generate: prefix-match hit


 inform inform office inform3v last/?3 only family only. need last has attend night office has should inv3 inform [v need will.v will r familyv night office only?? How should leave office3 be only? only3 in office r attend will should. attend be How office attend How last3 r be has3 should will should office [ night. office hasv in in in be last should night inform night only inform office be How family3v office inform leave bev be be attend has night be3 has. family last last office be last in family be office leave attend onlyvv will r only How in in attend in in leave office [ in only be attend. leave family be last only r be office How night office night will inv family How [ be. only How night only3. leave r [v only only office night family night [ last3 be only r [.. only leave only willv leave family r in r be leave night officev night family last bev How. r leave How nightv r How3 leavev office. in in will3 in last3 in in leave leave3 invv night family r nightv office r [ How r

Observation

* The response generated is lacking contextual relevance, exhibiting no meaning when compared with the user query having high randomness and token repetition.

###### Query 3: What should I do if I notice suspected harassment with my female colleague?

In [95]:
# Get the answer for user query
answer_f1_3, context_f1_3 = generate_context_based_response(query_3, max_tokens=256, temperature=1, top_p=0.5, top_k=90, repeat_penalty=0)
print(answer_f1_3)

Llama.generate: prefix-match hit


 file is by [ine with is/ with if/ I pen pen.ch my meant my harass use file/ do@ female file do3 is my notice only harass pen/ female3ine with is I female with is suspected is colle with only colle by colleinech only/ with suspected [ do use suspected file female only. my pen bych@ pench I [ my3@ pen suspected harass3 harass by pench file. by suspected@ is do colle@ I harass pen collech@33 mych my3 do use suspected. I. suspected@ine@ is do use3 harass is useineine I@ch by by my by is [ do@ is pen [ pen. do harass is by harass suspected is colle colle by harass harass colle do pen use3 doch@ collech. by3inech by3 suspected I.. I pen suspected harass I harass doch do@ by suspected colle pen3ine by my myine suspected by I by colle by [ine colle3 usech [ine my is my I I usech suspected is I by doch use. is harass suspected colle@ [ useine pen my harass my harass pen is do my I [ by by bych use [ use [ine use pen.


Observation

* The response is completely irrelevant and meaningless, characterized by extreme token looping and a total absence of grammatical or contextual logic.

##### Evaluation

###### Query 1: What are the effects on the benefits I receive if my probation is extended?

In [96]:
# Evaluating the user query
ground, rel = eval_grounded_relevance(question=query_1, answer=answer_f1_1, context=context_f1_1)
print("Groundedness Evaluation")
print(ground, end='\n\n')
print("Relevance Evaluation")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Groundedness Evaluation
 Based on the provided context, it's not possible to accurately determine an answer to the question due to the garbled and unclear nature of the AI-generated response. The metric of groundedness cannot be evaluated as there isn't a clear alignment between the generated claims and the source context text.

[0]: This rating signifies that no evaluation can be made based on the provided information.
[The given answer is not aligned with the question or context, making it impossible to evaluate its groundedness.]

Relevance Evaluation
 To evaluate the answer, I will first identify the main aspects of the question and then check how well they are addressed in the AI-generated response. The main aspects of the question are:

1. What happens to benefits when probation is extended?
2. Are there any specific rules or conditions for receiving benefits during an extension period?

Now, let's examine the answer and see if it covers these aspects:

The AI-generated response 

Observations

* For groundedness, the score was 0, since the response generated is not derived from the context also having numerous errors and is meaningless.
* For relevance, the generated response is not aligning with the metric criteria, yielding a score of 1 by not addressing the main aspects of the user query.

###### Query 2: There has been a demise in my family last night, and I need to attend the last rites. How should I inform the office, and will I be granted leave?

In [97]:
# Evaluating the user query
ground, rel = eval_grounded_relevance(question=query_2, answer=answer_f1_2, context=context_f1_2)
print("Groundedness Evaluation")
print(ground, end='\n\n')
print("Relevance Evaluation")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Groundedness Evaluation
 Based on the provided context, it is not possible to determine an accurate and grounded answer for the user's question about informing the office of a family member's demise and requesting leave due to the ambiguous nature of the AI-generated text. The following steps outline how one could verify the truthfulness of the answer against the context:

1. Identify specific claims made in the generated answer that can be aligned with information from the context.
2. Determine if each claim is strictly derived from the provided context or introduces outside knowledge, assumptions, or extrapolations.
3. Evaluate the extent to which the metric of groundedness is followed based on the alignment check and explicit steps above.

However, due to the lack of clear claims in the generated answer that can be directly linked with information from the context, it's impossible to provide a definitive evaluation or explanation for this scenario. Thus, I cannot rate nor explain th

Observations

* The score resulted for evaluating groundedness is 0, since the reponse is not derived from the context.
* The score resulted for evaluating relevance is 1, since the response is not adhered to the user's query providing some random tokens.

###### Query 3: What should I do if I notice suspected harassment with my female colleague?

In [98]:
# Evaluating the user query
ground, rel = eval_grounded_relevance(question=query_3, answer=answer_f1_3, context=context_f1_3)
print("Groundedness Evaluation")
print(ground, end='\n\n')
print("Relevance Evaluation")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Groundedness Evaluation
 Based on the provided context, there isn't a clear and explicit answer to the question regarding what actions should be taken if an employee notices suspected harassment with their female colleague. The context only outlines reporting channels, protection against retaliation, investigation timelines, and possible outcomes for confirmed violations.

To verify the truthfulness of any potential AI-generated answers, we would need to:
1. Identify specific actions recommended in the answer based on the information provided in the context.
2. Check if each action is strictly derived from the context without introducing assumptions or outside knowledge.
3. Ensure that no extrapolations are made beyond what's stated in the context.

However, since there isn't a clear and explicit answer in the context regarding specific actions to take when noticing suspected harassment with a female colleague, it would be difficult for an AI-generated answer to follow the groundedness

Observations

* The evaluation score for groundedness is 1, as the response generated is lacking any meaning, almost failing to extract information from context.
* The evaluation score for relevance is 1, the response generated is not adhered to the user's query, failing to address main aspects and not providing clear explanation.

**Key Observations**

* The above RAG model consistently failed to provide responses relevant to the user queries submitted by the employee.
* There was a token repetition in the response generated by the RAG model, since the repeat penalty parameter is set to 0.
* Also the responses generated by the model were having high degree of randomness, most probable reason might be the temperature parameter is set to 1.
* The model fails to provide diverse responses, since top_p parameter value is set to 0.5.
* When all of the responses were examined for groundedness, the evaluation was resulted between 0 and 1, since the response is not generating answers based on context due to the inconsistent values of parameters.
* When all of the responses were examined for relevance, the evaluation was resulted as 1. Since, the responses generated were not completely adhered to the user's query.
* Further fine tuning of hyper-parameters may improve the score of groundedness and relevance, which can be experimented subsequently.

#### Fine-tuning (Case: 2)
* max_tokens=512
* temperature=0.8
* top_p=0.8
* top_k=70
* repeat_penalty=0.5

###### Query 1: What are the effects on the benefits I receive if my probation is extended?

In [99]:
# Get the answer for user query
answer_f2_1, context_f2_1 = generate_context_based_response(query_1, max_tokens=512, temperature=0.8, top_p=0.8, top_k=70, repeat_penalty=0.5)
print(answer_f2_1)

Llama.generate: prefix-match hit


 I. effects on benefits for probation extended employees

1. benefits are only eligible for employees on probation if probation is successfullycompleted

2. if probation is extended, benefits only continue until the extended probation ends

3. benefits, if eligible, are only eligible for employees on probation if the probation is successfullycompleted, the extended probation is successfullycompleted,














































































































































































































































































































































































































































Observation

* Only some part of the answer generated by the model is relevant to the user query, also there are some tokens which were repetitve and meaningless.

###### Query 2: There has been a demise in my family last night, and I need to attend the last rites. How should I inform the office, and will I be granted leave?

In [100]:
# Get the answer for user query
answer_f2_2, context_f2_2 = generate_context_based_response(query_2, max_tokens=512, temperature=0.8, top_p=0.8, top_k=70, repeat_penalty=0.5)
print(answer_f2_2)

Llama.generate: prefix-match hit


 I. How to inform the office

* Verbal notification to the direct supervisor within 6 hours.
* Written/email notification within 24 hours.

II. leave granted

* granted for the death of the employee's immediate family.
* for the death of the employee's immediate family.

II. leave granted

* granted for the death of the employee's immediate family.
* for the death of the employee's immediate family.

II. leave granted

* granted for the death of the employee's immediate family.
* for the death of the employee's immediate family.

* Up to for the death of the employee's immediate family.

II. leave granted

* granted for the death of the employee's immediate family.
* for the death of the employee's immediate family.

* Up to for the death of the employee's immediate family.

* for the death of the employee's immediate family.

* for the death of the employee's immediate family.

* for the death of the employee's immediate family.

* for the death of the employee's immediate family.

* 

Observation

* As per the user query the answer generated by the model is relevant, but there are few sentences which were repetitive.

###### Query 3: What should I do if I notice suspected harassment with my female colleague?

In [101]:
# Get the answer for user query
answer_f2_3, context_f2_3 = generate_context_based_response(query_3, max_tokens=512, temperature=0.8, top_p=0.8, top_k=70, repeat_penalty=0.5)
print(answer_f2_3)

Llama.generate: prefix-match hit


 I. Report suspected harassment by using one of the following channels:

1. Confidential HR Helpline: +91-8000-456-789
2. Secure Email: HR.ethics@flykiteair.com
3. Anonymous Report: Anonymous Report-box

II. Report must include:

- Date
- Location
- Persons involved
- Brief incident description

II. Protection:

- Reporting employee: Protection against retaliation
- Report retaliation: Report retaliation against employee within 7-calendar-days

II. Reporting:

- Reporting: Report within 15-calendar-days
- Reporting: Reporting employee: Reporting employee retaliation within 7-calendar-days

II. Investigation:

- Report: Investigation within 7-days
- Report: Reporting employee: Reporting employee: Reporting employee: Reporting employee:
  Written-acknowledgement within 7-days
- Report: Reporting employee: Reporting employee: Reporting employee: Reporting employee:
  Report: Reporting employee: Reporting employee: Reporting employee: Reporting employee:
  Report: Reporting employee: Repor

Observation

* The answer is relevant to the user query, also there are few sentences which were irrelevant and are repeating.

##### Evaluation

###### Query 1: What are the effects on the benefits I receive if my probation is extended?

In [102]:
# Evaluating the user query
ground, rel = eval_grounded_relevance(question=query_1, answer=answer_f2_1, context=context_f2_1)
print("Groundedness Evaluation")
print(ground, end='\n\n')
print("Relevance Evaluation")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Groundedness Evaluation
 I. Evaluation of the generated answer for groundedness with respect to the context

1. Verification steps:
a. Identify if all claims in the answer are explicitly stated in the context.
b. Check if any assumptions or extrapolations have been made in deriving the answer.
c. Ensure that no outside knowledge has been introduced into the answer.

2. Alignment check:
a. The first claim "benefits are only eligible for employees on probation if probation is successfully completed" aligns with context point 3 under section 'Impact on Benefits, Seniority, and Contract'.
b. The second claim "if probation is extended, benefits only continue until the extended probation ends" aligns with no explicit statement in the context but can be inferred from points 2 and 3 under 'Impact on Benefits, Seniority, and Contract' as well as point 10 under 'Compensation and Benefits'.
c. The third claim "benefits, if eligible, are only eligible for employees on probation if the probation is

Observations

* The evaluation score for groundedness is 3, as some part of the response is generated from the context, while others require inference based on multiple points from the context.
* The evaluation score for relevance is 2, the generated response follows the metric criteria to a limited extent, by addressing only few aspects of the user query.

###### Query 2: There has been a demise in my family last night, and I need to attend the last rites. How should I inform the office, and will I be granted leave?

In [103]:
# Evaluating the user query
ground, rel = eval_grounded_relevance(question=query_2, answer=answer_f2_2, context=context_f2_2)
print("Groundedness Evaluation")
print(ground, end='\n\n')
print("Relevance Evaluation")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Groundedness Evaluation
 [5]: The metric is followed completely. Every single fact in the answer is explicitly backed by the context provided.

[EXPLANATION]: The AI-generated answer strictly adheres to the information presented in the context, providing accurate details on how to inform the office and the duration of leave granted for the death of an immediate family member.

Relevance Evaluation
 I. Evaluation Steps:
1. Identify if the answer covers both parts of the question: informing the office and leave granting.
2. Check if each part is addressed correctly based on the policy provided in context.
3. Ensure that all important aspects of the question are included in the answer.

II. Explanation:
The AI-generated answer covers both parts of the question, informing the office and leave granting. For the first part, it instructs to provide verbal notification within 6 hours and written/email notification within 24 hours. This aligns with the policy mentioned in context that employees

Observations

* The score resulted for evaluating groundedness is 5, as the response generated is adhered within the context.
* The score resulted for evaluating relevance is 3, as the metric is followed to a good extent, covering only few aspects of the user query.

###### Query 3: What should I do if I notice suspected harassment with my female colleague?

In [104]:
# Evaluating the user query
ground, rel = eval_grounded_relevance(question=query_3, answer=answer_f2_3, context=context_f2_3)
print("Groundedness Evaluation")
print(ground, end='\n\n')
print("Relevance Evaluation")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Groundedness Evaluation
 [5]: The metric is followed completely. Every single fact in the answer is explicitly backed by the context.
[EXPLANATION]: The AI-generated answer strictly adheres to the information provided in the context, without introducing any assumptions or outside knowledge. It accurately lists the reporting channels and requirements from the context, as well as the protection against retaliation and investigation timelines for both the initial harassment report and subsequent reports of retaliation. The AI-generated answer also correctly states that all reports must be made within specific timeframes and includes a reminder to report any retaliation within 7 calendar days. Overall, every detail in the generated answer is directly derived from the context text.

Relevance Evaluation
 I. To evaluate the answer, we need to check if it addresses all important aspects of the question regarding reporting suspected harassment with a female colleague at Flykite Airlines. The f

Observations

* For groundedness the evaluation score is 5, as the metric is followed completely, acurately listing contact information and procedure.
* For relevance the evaluation score is 3, since the response is adhered to the user query to a good extent but few tokens were irrelevant.

**Key Observations**

* The responses generated by the above RAG model has improved when compared with the previous model's response. But there are few sentences and tokens which were repeating and format is not that consistent
* The model's response for the user queries has some token repetition, since the repeat penalty parameter is set to 0.5.
* When compared with the previous model, the randomness in the generated response has been slightly improved due to the temperature parameter being set to 0.8.
* The model exhibits improved response diversity relative to the previous case, owing to the top_p parameter being set at 0.8.
* When all of the responses were examined for groundedness, the evaluation was resulted as 5 and 3, as two of the responses were mostly derived from the context, while one response requires multiple points from the context.
* When all of the responses were examined for relevance, the evaluation was resulted as 2 and 3. Since, the responses generated were adhered to the user's query to a limited extent, failing to address complete user query.
* Further fine tuning of hyper-parameters may improve the score of groundedness and relevance, which can be experimented subsequently.

#### Fine-tuning (Case: 3)
* max_tokens=512
* temperature=0.5
* top_p=0.8
* top_k=80
* repeat_penalty=0.8

###### Query 1: What are the effects on the benefits I receive if my probation is extended?

In [105]:
# Get the answer for user query
answer_f3_1, context_f3_1 = generate_context_based_response(query_1, max_tokens=512, temperature=0.5, top_p=0.8, top_k=80, repeat_penalty=0.8)
print(answer_f3_1)

Llama.generate: prefix-match hit


 I. Probation Extension and Benefits

A. Probation Extension Conditions

1. Achieved at least 60% of probationary objectives
2. A written PIP is issued
3. Both Department Head and HR Manager sign off

B. Benefits during Probation Extension

1. Annual leave encashment is not allowed
2. Internal role transfers are not allowed
3. Performance bonuses are not allowed
4. Seniority accrual starts only after successful probation completion

C. Benefits after Probation Extension

1. Benefits end on last working day unless law mandates otherwise
2. Employees are not eligible for benefits during probation extension

D. Probation Extension Duration

1. Extension for a maximum of 180 days
2. Extension may only be granted once

E. Performance Improvement Plan

1. Employees are required to have a Performance Improvement Plan during probation extension
2. Plan includes at least 3 measurable goals to be reviewed every 30 days
3. Failure to improve may result in termination or reassignment

F. Salary an

Observation

* The answer generated by the model is addressing the user query, also there are few sentences which were irrelevant.

###### Query 2: There has been a demise in my family last night, and I need to attend the last rites. How should I inform the office, and will I be granted leave?

In [106]:
# Get the answer for user query
answer_f3_2, context_f3_2 = generate_context_based_response(query_2, max_tokens=512, temperature=0.5, top_p=0.8, top_k=80, repeat_penalty=0.8)
print(answer_f3_2)

Llama.generate: prefix-match hit


 * Inform your direct supervisor verbally within 6 hours of the incident.
* Submit a death certificate, funeral notice, or obituary within 5 working days of returning to duty.
* Special leave of up to 5 consecutive working days will be granted for bereavement.
* Full-time employees are entitled to up to 5 days of bereavement leave per incident.
* The leave will be granted, provided the necessary documentation is provided.
* The leave will be deducted from the employee's annual leave entitlement.
* The employee's annual leave balance will be adjusted accordingly.
* The employee's annual leave balance will be communicated by the HR department.
* The employee's leave balance will be available on the HR portal.
* The employee will be able to check their leave balance by logging into the HR portal.
* The employee will be able to check their leave balance by contacting the HR department.
* The employee will be able to contact the HR department by emailing HR@flykiteairlines.com.
* The employ

Observation

* Few points of the response generated is addressing the user query, while few other points were repetitive and irrelevant.

###### Query 3: What should I do if I notice suspected harassment with my female colleague?

In [107]:
# Get the answer for user query
answer_f3_3, context_f3_3 = generate_context_based_response(query_3, max_tokens=512, temperature=0.5, top_p=0.8, top_k=80, repeat_penalty=0.8)
print(answer_f3_3)

Llama.generate: prefix-match hit


 1. Report the suspected harassment using the following channels:
   - Confidential HR Helpline: +91-8000-456-789
   - Secure Email Portal: hr.ethics@flykiteair.com
   - Anonymous drop-boxes in crew lounges and staff cafeterias

2. Reports must include:
   - Date
   - Location
   - Persons involved
   - Brief incident description

3. Protection against retaliation:
   - Report retaliation against the reporting employee within 7 calendar days.
   - HR will take immediate disciplinary action against retaliation.

4. Investigation & Resolution:
   - HR will initiate a preliminary review within 3 days.
   - Formal investigation will begin within 7 days.
   - HR will provide a written acknowledgment within 48 hours.
   - HR will provide a written outcome summary within 5 days.

5. Disciplinary actions for violations:
   - Possible outcomes for confirmed violations: written warning, mandatory sensitivity training, suspension, or termination.

6. Equal Opportunity and Non-Discrimination:
   -

Observation

* The answer generated is addressing the user query, having few irrelevant points and also there is some formatting issue.

##### Evaluation

###### Query 1: What are the effects on the benefits I receive if my probation is extended?

In [108]:
# Evaluating the user query
ground, rel = eval_grounded_relevance(question=query_1, answer=answer_f3_1, context=context_f3_1)
print("Groundedness Evaluation")
print(ground, end='\n\n')
print("Relevance Evaluation")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Groundedness Evaluation
 [5]: The answer strictly adheres to the context provided, as all facts stated in the answer can be traced back to specific information within it. No assumptions or outside knowledge have been introduced.

[EXPLANATION]: The AI-generated answer accurately reflects the conditions for probation extensions and their impact on benefits based on the context given. It follows each point mentioned in the context, including the requirements for achieving a probation extension, the restrictions on benefits during this period, and the duration of an extension. Additionally, it correctly states that seniority accrual starts only after successful probation completion and that employees are not eligible for benefits during the extension. The answer also mentions the Performance Improvement Plan (PIP) requirements and its role in evaluating employee performance during a probation extension. Overall, every single fact stated in the answer is explicitly backed by the context.



Observations

* For groundedness, the score resulted as 5, the metric is followed completely by deriving response from the context.
* For relevance, the generated response followed the metric criteria completely yielding the score of 5, as important aspects were addressed properly.

###### Query 2: There has been a demise in my family last night, and I need to attend the last rites. How should I inform the office, and will I be granted leave?

In [109]:
# Evaluating the user query
ground, rel = eval_grounded_relevance(question=query_2, answer=answer_f3_2, context=context_f3_2)
print("Groundedness Evaluation")
print(ground, end='\n\n')
print("Relevance Evaluation")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Groundedness Evaluation
 3: The metric is followed to a good extent

Explanation:
The AI-generated answer primarily relies on the context provided, which includes information about bereavement leave and documentation requirements for special leaves at Flykite Airlines. However, there are some claims in the generated answer that go beyond what's explicitly stated in the context. For instance, the statement "Special leave of up to 5 consecutive working days will be granted for bereavement" is an assumption based on the information given about full-time employees being entitled to up to 5 days of bereavement leave per incident. Additionally, there are several statements regarding various ways to contact HR and check leave balances that aren't explicitly mentioned in the context but could still be considered reasonable assumptions within the given organizational structure. Overall, while some parts of the generated answer extend beyond what's strictly stated in the context, it remains grou

Observations

* The score resulted for evaluating groundedness is 3, since there are some claims in the generated reponse which goes beyond the context.
* The score resulted for evaluating relevance is 5, as the response generated is completely adhered with the user query.

###### Query 3: What should I do if I notice suspected harassment with my female colleague?

In [110]:
# Evaluating the user query
ground, rel = eval_grounded_relevance(question=query_3, answer=answer_f3_3, context=context_f3_3)
print("Groundedness Evaluation")
print(ground, end='\n\n')
print("Relevance Evaluation")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Groundedness Evaluation
 [3]: The answer follows the metric to a good extent as it primarily relies on the provided context and outlines the steps an employee should take when they notice suspected harassment with their female colleague. However, some parts of the answer (e.g., "Flykite Airlines strictly prohibits harassment or unfair treatment based on gender") are already stated in the context but are repeated without explicit reference to it.

[EXPLANATION]: The generated answer is primarily derived from the provided context and outlines the reporting channels, required report content, protection against retaliation, investigation & resolution timelines, disciplinary actions for violations, and equal opportunity policy. However, some parts of the answer are repeated without explicit reference to their source in the context. For instance, "Flykite Airlines strictly prohibits harassment or unfair treatment based on gender" is already stated in the context under point 1. Zero-Tolerance

Observations

* For groundedness the evaluation score is 3, since the metric is followed to a good extent, however, certain lines—like the zero-tolerance policy for gender harassment—were repeated in the response without explicitly referencing their original source in the context.
* For relevance the evaluation score is 3, since the response is failing to address details about the consequences and requires some improvement.

**Key Observations**

* When compared with the previous case the responses generated by the model is relevant, addressing most part of the user query.
* The randomness observed in the model's responses has been reduced when compared with the previous models, due to the temperature parameter being set to 0.5.
* When all of the responses were examined for groundedness, the evaluation was resulted as 3 and 5, While one response adheres completely to the user query, the other two responses either go beyond the context or repeat sentences without explicit references with the context.
* When all of the responses were examined for relevance, the evaluation was resulted as 3 and 5, while two responses were adhered with the user query and one response is failing to address few aspects of the user query.
* Further fine tuning of hyper-parameters may improve the score of groundedness and relevance, which can be experimented subsequently.

#### Fine-tuning (Case: 4)
* max_tokens=1024
* temperature=0.2
* top_p=0.95
* top_k=80
* repeat_penalty=1

###### Query 1: What are the effects on the benefits I receive if my probation is extended?

In [111]:
# Get the answer for user query
answer_f4_1, context_f4_1 = generate_context_based_response(query_1, max_tokens=1024, temperature=0.2, top_p=0.95, top_k=80, repeat_penalty=1)
print(answer_f4_1)

Llama.generate: prefix-match hit


 * Benefits, such as health insurance and retirement plan, only start accruing after successful probation completion.
* During the probation period (including extensions), employees are not eligible for annual leave encashment, internal role transfers, or performance bonuses.
* Seniority accrual also starts only after successful probation completion.
* If an employee has two or more extensions in different roles, HR will assess contract renewal eligibility based on performance history.
* There is no information provided regarding the impact of probation extensions on specific benefits, such as health insurance or retirement plans, beyond the points mentioned above.

I don't know.


Observation

* The response generated by the model directly adheres to the user query; however, the inclusion of the statement "There is no information provided" and "I don't know" introduces a structural contradiction in the user experience and breaks the intended operational persona.

###### Query 2: There has been a demise in my family last night, and I need to attend the last rites. How should I inform the office, and will I be granted leave?

In [112]:
# Get the answer for user query
answer_f4_2, context_f4_2 = generate_context_based_response(query_2, max_tokens=1024, temperature=0.2, top_p=0.95, top_k=80, repeat_penalty=1)
print(answer_f4_2)

Llama.generate: prefix-match hit


 * Inform your direct supervisor verbally within 6 hours of the incident.
* Submit a written or email notification within 24 hours.
* Special leave is granted for bereavement up to 5 consecutive working days per incident.
* Ensure that the documentation (death certificate, funeral notice, or obituary) is submitted within 5 working days of returning to duty.


Observation

* The response generated is adhered to the user query, mostly addressing all the aspects.

###### Query 3: What should I do if I notice suspected harassment with my female colleague?

In [113]:
# Get the answer for user query
answer_f4_3, context_f4_3 = generate_context_based_response(query_3, max_tokens=1024, temperature=0.2, top_p=0.95, top_k=80, repeat_penalty=1)
print(answer_f4_3)

Llama.generate: prefix-match hit


 1. Report the incident to the Confidential HR Helpline: +91-8000-456-789 or use the Secure Email Portal: hr.ethics@flykiteair.com.
2. Provide details such as date, location, persons involved, and a brief description of the incident.
3. Protect your colleague from retaliation by reporting any alleged retaliation within 7 calendar days of occurrence.
4. HR will initiate a preliminary review within 3 working days and begin a formal investigation within 7 working days.
5. Both the complainant and accused will receive written acknowledgment of the complaint within 48 hours and an outcome summary and next steps within 5 working days of conclusion.
6. Possible outcomes for confirmed violations include written warnings, mandatory sensitivity and workplace conduct training, suspension, or immediate termination.
7. If you are unsure, contact the HR department for clarification.

I don't know: penchala.vineeth3@gmail.com
R02IHZ4C3E
This file is meant for personal use only.


Observation

* The response generated by the model is adheresd to the user query. Almost providing every detail, but failing to act as a HR persona since in the response it mentioned as "I don't know" and few other sentences which were irrelevant.

##### Evaluation

###### Query 1: What are the effects on the benefits I receive if my probation is extended?

In [114]:
# Evaluating the user query
ground, rel = eval_grounded_relevance(question=query_1, answer=answer_f4_1, context=context_f4_1)
print("Groundedness Evaluation")
print(ground, end='\n\n')
print("Relevance Evaluation")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Groundedness Evaluation
 [1]: The answer does not strictly follow the context since it makes a claim about the potential impact on specific benefits (health insurance and retirement plan) that is not explicitly stated in the provided context. While the context mentions that certain benefits are not available during probation, it does not specify whether this applies to extensions as well or how these benefits change upon successful completion of an extension. Therefore, the answer introduces assumptions based on outside knowledge (the general understanding that benefits may be affected by employment status changes) and extrapolates from the information given in the context without sufficient justification.

Relevance Evaluation
 [3]: The AI-generated answer follows the metric to a good extent by addressing the main aspects of the question related to the impact of probation extensions on benefits such as annual leave encashment, internal role transfers, performance bonuses, and seniorit

Observations

* The evaluation score for groundedness is 1, as the response introduced assumptions and outside knowledge regarding the impact on health insurance and retirement plans that were not explicitly stated in the provided context.
* The evaluation score for relevance is 3, the generated response could have improved by addressing few more aspect when compared with the user query.

###### Query 2: There has been a demise in my family last night, and I need to attend the last rites. How should I inform the office, and will I be granted leave?

In [115]:
# Evaluating the user query
ground, rel = eval_grounded_relevance(question=query_2, answer=answer_f4_2, context=context_f4_2)
print("Groundedness Evaluation")
print(ground, end='\n\n')
print("Relevance Evaluation")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Groundedness Evaluation
 [3]: The answer follows the metric to a good extent as it directly references and advises actions based on information from the context regarding bereavement leave procedures. However, there are some minor discrepancies:

1. Verbal notification within 6 hours is mentioned in the context for emergencies but not explicitly stated for bereavement leaves in the answer. The answer only mentions verbal notification for incidents and does not specify if it applies to all leave types or just emergencies.
2. The documentation submission deadline (within 5 working days of returning to duty) is mentioned correctly, but there's no need to explicitly state that this information comes from the context since it's already provided in the question itself.
3. The answer does not mention any limitations on the number of consecutive bereavement leave days for employees under probation or those under harassment/discrimination investigation as stated in the context, but these condit

Observations

* The score resulted for evaluating groundedness is 3, as the response generated is adhered to the context at good extent, but there are few assumptions.
* The score resulted for evaluating relevance is 3, as the metric is followed to a good extent addressing few aspects of the user query.

###### Query 3: What should I do if I notice suspected harassment with my female colleague?

In [116]:
# Evaluating the user query
ground, rel = eval_grounded_relevance(question=query_3, answer=answer_f4_3, context=context_f4_3)
print("Groundedness Evaluation")
print(ground, end='\n\n')
print("Relevance Evaluation")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Groundedness Evaluation
 [5]: The answer follows the metric completely as every single fact in the answer is explicitly backed by the context provided.
[EXPLANATION]: The AI-generated answer strictly adheres to the information given in the context, providing step-by-step instructions on how to report suspected harassment based on the available reporting channels and procedures outlined in the text. Additionally, it aligns with the consequences for retaliation, investigation timelines, and possible outcomes for confirmed violations mentioned in the context. The answer does not introduce any assumptions or outside knowledge.

Relevance Evaluation
 [5]: The AI-generated answer directly and comprehensively addresses the question by providing specific steps that align with the provided context regarding reporting suspected harassment, protecting against retaliation, investigation timelines, and potential disciplinary actions. It covers all important aspects of the question and follows the m

Observations

* For groundedness the evaluation score is perfect 5, as the response generated is adhered within the context.
* For relevance the evaluation score is 5, since the response is completely adhered to the user query.

**Key Observations**

* The model has improved significantly when compared with the previous case addressing most part of the user query.
* The model's response was free of token repetition, since the repeat penalty parameter is set to 1.
* The significant reduction in response randomness, relative to the previous model, is directly attributable to adjusting the temperature parameter to a very low value of 0.2.
* When all of the responses were examined for groundedness, the evaluation was resulted as 1, 3 and 5. Since one response is completely derived from the context, while other two responses showed some inconsistencies by generating response based on assumptions.
* When all of the responses were examined for relevance, the evaluation was resulted as 3 and 5. Only one response is completely adhered to the user query, while other two responses showed some discrepancies failing to address few aspects of the query.
* Further fine tuning of hyper-parameters may improve the score of groundedness and relevance, which can be experimented subsequently.

#### Fine-tuning (Case: 5)
* max_tokens=1024
* temperature=0
* top_p=0.95
* top_k=50
* repeat_penalty=1.2

###### Query 1: What are the effects on the benefits I receive if my probation is extended?

In [117]:
# Get the answer for user query
answer_f5_1, context_f5_1 = generate_context_based_response(user_query=query_1, max_tokens=1024, temperature=0, top_p=0.95, top_k=50, repeat_penalty=1.2)
print(answer_f5_1)

Llama.generate: prefix-match hit


 * Benefits, such as health insurance and retirement plan enrollment, do not accrue during the probation period.
* After successful completion of probation, seniority accrual starts for benefit eligibility purposes.
* If your probation is extended, benefits will continue until the last working day of your employment unless mandated by law otherwise.


Observation

* The answer generated by the model is addressing the user query by explaining every aspect.

###### Query 2: There has been a demise in my family last night, and I need to attend the last rites. How should I inform the office, and will I be granted leave?

In [118]:
# Get the answer for user query
answer_f5_2, context_f5_2 = generate_context_based_response(user_query=query_2, max_tokens=1024, temperature=0, top_p=0.95, top_k=50, repeat_penalty=1.2)
print(answer_f5_2)

Llama.generate: prefix-match hit


 * Inform your direct supervisor verbally within 6 hours of the incident.
* Submit written or email notification within 24 hours.
* Special leave is granted for bereavement up to a maximum of 5 consecutive working days per incident.


Observation

* The response generated addressed the user query appropriately, explaining the complete procedure and acting more like an HR persona.

###### Query 3: What should I do if I notice suspected harassment with my female colleague?

In [119]:
# Get the answer for user query
answer_f5_3, context_f5_3 = generate_context_based_response(user_query=query_3, max_tokens=1024, temperature=0, top_p=0.95, top_k=50, repeat_penalty=1.2)
print(answer_f5_3)

Llama.generate: prefix-match hit


 1. Report the incident using one of the following channels:
   - Confidential HR Helpline: +91-8000-456-789 (available Mon–Sat, 9 AM–8 PM IST)
   - Secure Email Portal: hr.ethics@flykiteair.com
   - Anonymous drop-boxes in crew lounges and staff cafeterias
2. Include the following details when reporting: date, location, persons involved, brief incident description
3. Protection against retaliation applies to you as well; report any retaliatory actions within 7 calendar days of occurrence.


Observation

* The answer generated by the model is addressing the user query appropriately.

##### Evaluation

###### Query 1: What are the effects on the benefits I receive if my probation is extended?

In [120]:
# Evaluating the user query
ground, rel = eval_grounded_relevance(question=query_1, answer=answer_f5_1, context=context_f5_1)
print("Groundedness Evaluation")
print(ground, end='\n\n')
print("Relevance Evaluation")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Groundedness Evaluation
 5: The metric is followed completely.
[EXPLANATION]: The answer strictly adheres to the information provided in the context. It states that employees do not accrue benefits during probation and that seniority accrual starts after successful completion of probation. Additionally, it correctly notes that if an employee's probation is extended, their benefits will continue until the last working day of their employment unless mandated by law otherwise. The answer does not introduce any assumptions or outside knowledge.

Relevance Evaluation
 5: The metric is followed completely.
[EXPLANATION]: The AI-generated answer directly addresses the main aspects of the question regarding the effects on benefits during and after probation extension. It mentions that benefits do not accrue during the probation period, seniority accrual starts upon successful completion of probation, and benefits continue until the last working day of employment when extended unless mandated b

Observations

* For groundedness, the score resulted as 5 the metric is followed completely, where the response is derived from context.
* For relevance, the generated response followed the metric completely, yielding the score of 5. Addressed every aspect of the query.

###### Query 2: There has been a demise in my family last night, and I need to attend the last rites. How should I inform the office, and will I be granted leave?

In [121]:
# Evaluating the user query
ground, rel = eval_grounded_relevance(question=query_2, answer=answer_f5_2, context=context_f5_2)
print("Groundedness Evaluation")
print(ground, end='\n\n')
print("Relevance Evaluation")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Groundedness Evaluation
 Based on the provided context, here are the steps needed to verify the truthfulness of the answer:

1. The employee should inform their direct supervisor verbally within 6 hours of the incident as stated in the context under "Notification Process" section for sudden emergencies.
2. Within 24 hours, they must submit written or email notification about the leave request as mentioned in the same section.
3. According to the "Special Leave Policy — Flykite Airlines," special leave is granted for bereavement up to a maximum of 5 consecutive working days per incident under the covered situations and entitlement & duration limits sections.

The generated answer matches the context, as it includes both steps required by the company policy for reporting an emergency leave request (verbal notification within 6 hours and written/email notification within 24 hours) and the maximum number of days granted for bereavement special leave according to Flykite Airlines' policies.

Observations

* The score resulted for evaluating groundedness is 3, as the metric followed to good extent.
* The score resulted for evaluating relevance is 5, as the response generated is completely adhered with the user query by addressing every aspect.

###### Query 3: What should I do if I notice suspected harassment with my female colleague?

In [122]:
# Evaluating the user query
ground, rel = eval_grounded_relevance(question=query_3, answer=answer_f5_3, context=context_f5_3)
print("Groundedness Evaluation")
print(ground, end='\n\n')
print("Relevance Evaluation")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Groundedness Evaluation
 [5]: The answer is rated a 5 because every single fact in the answer is explicitly backed by the context provided. The steps for reporting suspected harassment are directly taken from the accepted reporting channels listed in the context, and the protection against retaliation information is also sourced from the context.

[EXPLANATION]: The AI-generated answer strictly adheres to the groundedness metric as it only includes facts that can be found within the provided context. No assumptions or outside knowledge were introduced into the response.

Relevance Evaluation
 To evaluate the answer's relevance to the question, we need to check if it addresses all important aspects of the question. The main parts of the question are:

1. What should I do if I notice suspected harassment with my female colleague?
2. Accepted reporting channels and required information for reports.
3. Protection against retaliation.

The answer directly addresses these points by suggestin

Observations

* For groundedness the evaluation score is 5, since the metric is followed completely by generating the response based on the context.
* For relevance the evaluation score is 5, since the response is adhered to the user query and addressed every aspect.

**Key Observations**

* When compared with all the previous cases the responses generated by the model is relevant, addressing every aspect of the user query.
* When all of the responses were examined for groundedness, the evaluation was resulted as 3 and 5, as the responses were mostly derived from the context, while only one response generated is showing some inconsistencies.
* When all of the responses were examined for relevance, the evaluation was resulted as 5. All the responses were completely adhered to the user query.
* This indicates that the response is grounded to the context and is relevant to the user query. Therefore, both the retrieval and augmentation is perfect.
* The combination of perfect groundedness and relevance ensures the user receives an answer that is both correct(grounded) and useful(relevant), maximizing the utility of the RAG system.

## Insights and Recommendations

### Insights

* **Question Answering using LLM**: Out of three user queries, for one of the reponse generated by the llm is hallucinating between two domains i.e., criminal and corporate due to a particular term i.e., Probation, used in the user query. Also, the llm is responding generically not acting like a company bot.
* **Question Answering using LLM and Prompt Engineering**: Cross-domain hallucinations have been eliminated by the response produced by the system prompt with user query, functioning more like a corporate HR assistant. while continuing to give basic answers without consulting corporate papers.
* **Evaluation**
    * **Groundedness**: This refers to the degree to which the generated response is grounded within the context provided. But here the responses generated by both the llm and llm with prompt engineering received a very low score of 1 because the context was not provided.
    * **Relevance**: This refers to the degree to which the generated response is relevant to the user query. Here for most the responses generated by both the llm and llm with prompt engineering received a score between 4 and 5 indicating that they generally addressed the user's query effectively.
* **Data Preparation for RAG**
    * **Chunking**: Different chunking methods were used i.e., character, recursive character and semantic text splitters were used, where semantic method outperformed, since the document is chunking based on similarity between adjacent sentences, while character text splitter is splitting the word abruptly having no meaning and recursive character text splitter is fixed splitting but not splitting the word abrutly.
    * **Embedding**: Three different embedding models i.e., 'thenlper/gte-large', 'BAAI/bge-large-en-v1.5' and 'nomic-ai/nomic-embed-text-V1.5' were used to convert the sentences into dimensional vectors.
    * **Vector Store**: 'Chroma' database is used to store the document chunks in embeddings to perform fast similarity searches.
    * **Retriever**: User query is converted to embeddings, so that relevant document chunk embeddings stored in the database were retrieved using 'similarity' method.
* **Question Answering using RAG**: The introduction of context and restricting the LLM to answer user query based on the context levaraged the model to perform efficiently when compared with the LLM and LLM with prompt engineering. Different combination of parameters has fine-tuned and it has been observed that:
    * The model is failing to generate complete response because of restricting the max_tokens parameter.
    * The more the temperature value the more the randomness has been reflected in the generated response.
    * The lower the repeat_penalty value the more the tokens get repeated in the response generated by the model.
    * The higher the value of top_p resulted in more diverse response, while a lower value has resulted in less diverse response.
    * Low top_k restricts the model to only the few most probable next tokens, resulting in more focused and predictable text, whereas high top_k expands the selection to a larger pool of likely tokens, allowing for greater diversity in the generated output.
* **Evaluation**
    * **Groundedness**: As the context is exposed and the responses generated by the RAG model with the right combination of parameters has performed perfectly well with the rating ranging from 4 to 5.
    * **Relevance**: The score resulted is oerfect 5, since the responses were completely adhered to the user query, addressing every key aspect.

### Recommendations

* **Integrate Contextual Retrieval for LLM Responses**: The low groundedness scores and the generic nature of responses can be overcome by exposing relevant context to the llm.
* **Refine Prompt Engineering with Retrieved Context**: While prompt engineering improved domain adherence, it can be further enhanced by incorporating the retrieved context directly into the prompt. The system prompt should guide the LLM to always base its answers only on the provided context.
* **Better Embedding Model**: When all the embedding models were compared by examining the retrieved chunks as the user query, 'BAAI/bge-large-en-v1.5' embedding model out-performed followed by 'thenlper/gte-large' and 'nomic-ai/nomic-embed-text-V1.5'. Therefore, 'BAAI/bge-large-en-v1.5' embedding model is used to convert chunks into dimensional vectors.
* **Optimal k value**: As the document is chunked based on semantic method, therefore an optimal value of k=4 is recommended so that only relevant chunks were retrieved based on user query.
* **Optimum parameters for RAG model**: In this case, the RAG model is tested for different combinations and it is recommended to use the right combination of parameters that are used to address the user query for this particular context as stated below:
  * max_tokens=1024
  * temperature=0
  * repeat_penalty=1.2
  * top_p=0.95
  * top_k=50
* **Deployment**: This RAG model is recommended to deploy, so that HR's can focus on more important work instead of answering almost the same query raised by different employee. Also, it would be very helpful for any dynamic changes in the policy.
